#### Imports

In [ ]:
import numpy as np
from itertools import chain, combinations 
import matplotlib.pyplot as plt
from scipy.optimize import fsolve
from math import comb
import gurobipy as gp
from gurobipy import GRB
import cplex
import xpress as xp
import scipy.sparse as sp
import plotly.graph_objects as go
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import scipy


#### Preliminary functions for defining objective $T(\mathbf{O})$

In [ ]:
     
def find_subsets(n):
    #This function outputs the collection of subsets of $n$ individuals, 
    # also with the size of the set, N, and a few filtered versions of 
    # X for quickness of the code
    X = list(map(set, chain.from_iterable(combinations(range(n), r) for r in range(2, n+1))))
    N = len(X)
    X_sizes_increasing = [[x for x in X if len(x)<=k] for k in range(2, n+1)]
    X_sizes_decreasing = [[x for x in X if len(x)>=k] for k in range(2, n+1)]
    return X, N, X_sizes_increasing, X_sizes_decreasing

def S(c, X, X_sizes_decreasing, N):
    #Used for defining $T$. Set defined in main text.
    interaction = X[c]
    size = len(interaction)
    needed_X = X_sizes_decreasing[size-2]
    c_vals = [X.index(set_) for set_ in needed_X if interaction.issubset(set_)]
    return c_vals

def B(c, X, X_sizes_increasing, N): 
    #Used for defining $T$. Set defined in main text.
    interaction = X[c]
    size = len(interaction)
    needed_X = X_sizes_increasing[size-2]
    c_vals = [X.index(set_) for set_ in needed_X if set_<interaction]
    return c_vals

def J(i, S_sets):
    #Used for constructing $l$, as described in supporting information
    c_vals = [S_sets.index(set_) for set_ in S_sets if i in set_] 
    return c_vals

def Z(i, j, S_sets, B_sets):
    #Used for constructing $Q_1$, as described in supporting information
    S_j = set([c for c in S_sets[j] if c!=j])
    B_i = B_sets[i].union({i})
    return B_i.intersection(S_j)
    
def V(i, j, B_sets):
    #Used for constructing $Q_2$, as described in supporting information
    B_i = B_sets[i].union({i})
    B_j = B_sets[j].union({j})
    V_ij = B_i.intersection(B_j)
    return V_ij

def give_sums_and_prods(X, N, N_cells):
    #Precompute the sums and products needed throughout $T$
    sum_Ns = np.zeros(N)
    prod_Ns = np.ones(N)

    for c in range(N):
        for k in X[c]:
            sum_Ns[c] += N_cells[k]
            prod_Ns[c] *= N_cells[k]
    return sum_Ns, prod_Ns
            
def get_l(N, S_sets, sum_Ns, prod_Ns):
    #Compute the vector $l$ as described in supporting information
    l = np.zeros(N)
    for i in range(N):
        for c in J(i, S_sets):
            l[i] += (sum_Ns[c] / prod_Ns[c])
    return l

def get_M(X, N, S_sets, B_sets, prod_Ns):
    #Compute the matrix $M$ as described in supporting information
    Q1 = np.zeros([N, N])
    Q2 = np.zeros([N, N])
    #Get $Q1$ for the construction of $T$
    # Quite lazy coding here, but works well enough. 
    # Don't need the condition as we have to multiply i=j entries by 2 anyway.
    for i in range(N):
        for j in range(i, N):
            for c in V(i, j, B_sets):
                interaction = X[c]
                f_c = len(interaction)
                Q1[i, j] += 2*f_c / prod_Ns[c]
            Q1[j, i] = Q1[i, j] #Ensuring Q1 is diagonal

        for j in range(i+1, N): #We know i=j will have zero contribution, so ignore this case
            if X[i] < X[j]: #Reverse condition not needed because of the iteration ordering
                g_i = len(X[i]) - 1
                for c in Z(j, i, S_sets, B_sets):
                    Q2[i, j] += g_i / prod_Ns[c]      
            Q2[j, i] = Q2[i, j] #Ensuring Q1 is diagonal
    return -(Q1 + Q2)


#### Other useful functions

In [ ]:
def get_shared_and_unique(O, n, N_cells, X):
    #Get the proportion of uniquely known and shared area for each individual, from spatial structure O
    shared_prop = np.zeros(n)
    unique_prop = np.zeros(n)
    for k in range(n):
        all_with_k = [X.index(set_) for set_ in X if {k}.issubset(set_)]
        unique_cells = N_cells[k] - sum([O[i] for i in all_with_k])
        unique_prop[k] = unique_cells / N_cells[k]
        shared_prop[k] = 1 - unique_prop[k]
    return shared_prop, unique_prop

def get_w_i_n(X, O, N_cells, N, i, n):
    #Compute w_n^i, as defined in main text
    if i >= 2 and i<= n:
        g_vals = [len(X[c])-1 for c in range(N)]
        sum_of_areas = sum(N_cells)
        overcounted_area = sum([g_vals[c]*O[c] for c in range(N)])
        lower_limit = sum([comb(n, j) for j in range(2, i)]) + 1
        upper_limit = lower_limit - 1 + comb(n, i)
        order_i_shared = sum(O[lower_limit-1:upper_limit]) #Shifting for python ordering 
        w = order_i_shared / (sum_of_areas - overcounted_area)
    else: 
        w = 0
    return w
  
def get_T(O, n, N_cells):
    #Compute the value of objective function T at spatial structure O
    #O is an array of the overlap values. 
    #n is the number of agents

    O = np.array(O)
    X, N, X_sizes_increasing, X_sizes_decreasing = find_subsets(n)
    S_sets= [set(S(c, X, X_sizes_decreasing, N)) for c in range(N)]
    B_sets = [set(B(c, X, X_sizes_increasing, N)) for c in range(N)]
    sum_Ns, prod_Ns = give_sums_and_prods(X, N, N_cells)
            
    l = get_l(N, S_sets, sum_Ns, prod_Ns)
    M = get_M(X, N, S_sets, B_sets, prod_Ns)
    
    T = (0.5* O.T @ M @ O + l.T @ O) 
    return T

def get_w_weight_mean(n, ws):
    #Compute the weighted mean of the ws, including the order-1 weight. This is defined as K in the main text.
    ws_with_ones = np.zeros(n)
    ws_with_ones[1:] = ws
    ws_with_ones[0] = 1-np.sum(ws_with_ones)
    w_mean = sum([ws_with_ones[i]*(i+1) for i in range(n)]) 
    return w_mean


## Scenario 1: Homogeneous foraging abilities

#### Without foraging constraint

##### Optimisation code

In [ ]:

def HOMO_INTEGER_OPTIMISE_gp(n, N_cell_each):
    #Optimise in the homogeneous case with n individuals with N_cell_each knowledge capacities. No foraging constraint. Using Gurobi solver.
    with gp.Env(empty=True) as env: #To avoid text showing the current optimisation procedure. 
        env.setParam('OutputFlag', 0)
        env.start()
        
        #Initialise the model
        m = gp.Model(env = env, name = "test")
        
        #Define the variables (the overlapping points)
        X, N, X_sizes_increasing, X_sizes_decreasing = find_subsets(n)
        
        O = m.addMVar(shape=N, vtype=GRB.INTEGER, name="O") #Constraining the solutions to be integers
    
        #We define the points, and then the environment determines the geography - that's the argument
        
        #Define the objective 
        N_cells = N_cell_each*np.ones(n)
        S_sets= [set(S(c, X, X_sizes_decreasing, N)) for c in range(N)]
        B_sets = [set(B(c, X, X_sizes_increasing, N)) for c in range(N)]
        sum_Ns, prod_Ns = give_sums_and_prods(X, N, N_cells)
        
        
        l = get_l(N, S_sets, sum_Ns, prod_Ns)
        M = get_M(X, N, S_sets, B_sets, prod_Ns)
        
        m.setObjective(l@O + 0.5 * O.T @ M @ O, GRB.MAXIMIZE)
        
        #Define the constraints
        
        m.addConstrs((O[i] >= 0 for i in range(N)), name = 'non-neg bounds') #For non-negativity
        
        N_diff = (2**(n-1)) - 1
        entries = np.ones(n*N_diff)
        row_indices = np.zeros(n * N_diff)
        col_indices = [None] * n * N_diff
        
        for i in range(n):
            col_indices[i*N_diff:(i+1)*N_diff] = [X.index(set_) for set_ in X if {i}.issubset(set_)]
            row_indices[i*N_diff:(i+1)*N_diff] = i 
            
        col_indices = np.array(col_indices)
        
        G = sp.csr_matrix((entries, (row_indices, col_indices)), shape=(n, N))
        
        h = np.array(N_cells)
        
        m.addConstr(G @ O <= h, name='sharing') #Defining the "don't share too much" constraint
        
        m.optimize()

    return O.X, m.ObjVal

def HOMO_INTEGER_OPTIMISE_xp(n, N_cell_each):
    #Optimise in the homogeneous case with n individuals with N_cell_each knowledge capacities. No foraging constraint. Using Xpress solver.
    xp.controls.outputlog = 0
        
    #Initialise the model
    m = xp.problem(name = "Xpress model")
    
    #Define the variables (the overlapping points)
    X, N, X_sizes_increasing, X_sizes_decreasing = find_subsets(n)


    O = np.array([m.addVariable(lb=0, vartype = xp.integer, name = f'O_{i+1}') for i in range(N)])
    
    #We define the points, and then the environment determines the geography - that's the argument
    
    #Define the objective 
    N_cells = N_cell_each*np.ones(n)
    S_sets= [set(S(c, X, X_sizes_decreasing, N)) for c in range(N)]
    B_sets = [set(B(c, X, X_sizes_increasing, N)) for c in range(N)]
    sum_Ns, prod_Ns = give_sums_and_prods(X, N, N_cells)
    
    l = get_l(N, S_sets, sum_Ns, prod_Ns)
    M = get_M(X, N, S_sets, B_sets, prod_Ns)
    
    m.setObjective(l@O + 0.5 * O.T @ M @ O, sense = xp.maximize)
    
    #Define the constraints

    index_sets = [[X.index(set_) for set_ in X if {i}.issubset(set_)] for i in range(n)]

    for i in range(n):
        m.addConstraint(xp.Sum(O[index_sets[i]]) <= N_cells[i])    
    
    m.optimize()

    O_optim = np.array(m.getSolution())


    return np.round(O_optim), l@O_optim + 0.5 * O_optim.T @ M @ O_optim

def HOMO_INTEGER_OPTIMISE_cplex(n, N_cell_each):
    #Optimise in the homogeneous case with n individuals with N_cell_each knowledge capacities. No foraging constraint. Using CPLEX solver.

    #Initialise the model
    m = cplex.Cplex() 
    m.set_problem_type(m.problem_type.MIQP)
    
    #Define the variables (the overlapping points)
    X, N, X_sizes_increasing, X_sizes_decreasing = find_subsets(n)

    
    m.objective.set_sense(m.objective.sense.maximize)
        
    #Define the objective 
    N_cells = N_cell_each*np.ones(n)
    S_sets = [set(S(c, X, X_sizes_decreasing, N)) for c in range(N)]
    B_sets = [set(B(c, X, X_sizes_increasing, N)) for c in range(N)]

    sum_Ns, prod_Ns = give_sums_and_prods(X, N, N_cells)
    
    
    l = get_l(N, S_sets, sum_Ns, prod_Ns)
    M = get_M(X, N, S_sets, B_sets, prod_Ns)

    O = m.variables.add(obj=l, types=[m.variables.type.integer] * N, lb = [0] * N) #Here l is setting the linear coefficient of each variable in the objective

    m.objective.set_quadratic([cplex.SparsePair(ind = range(N), val = [M[i, k] for k in range(N)]) for i in range(N)])

    #Define the constraints
    index_sets = [[X.index(set_) for set_ in X if {i}.issubset(set_)] for i in range(n)]
    
    m.linear_constraints.add(
        lin_expr = [cplex.SparsePair(ind = index_sets[i], val = [1] * len(index_sets[i])) for i in range(n)], 
    senses = ["L"] * n,
    rhs = N_cells) #Defining the "don't share too much" constraint


    #
    #m.add_constraints(np.Sum(O[index_sets[i]]) <= N_cells[i] for i in range(n)) #Defining the "don't share too much" constraint
    m.parameters.optimalitytarget.set(3)

    #Hiding all of the output
    m.set_results_stream(None)
    m.set_log_stream(None)

    m.solve()

    O_optim = m.solution.get_values()

    return np.round(O_optim), m.solution.get_objective_value()

def HOMO_INTEGER_OPTIMISE_best(n, N_cell_each):
#Optimise with each solver and return the best solution
    T_vals = [None] * 3
    optims = [None] * 3
    solver = ['G', 'X', 'C']

    optims[0], T_vals[0] = HOMO_INTEGER_OPTIMISE_gp(n, N_cell_each)
    optims[1], T_vals[1] = HOMO_INTEGER_OPTIMISE_xp(n, N_cell_each)
    optims[2], T_vals[2] = HOMO_INTEGER_OPTIMISE_cplex(n, N_cell_each)

    best_index = T_vals.index(max(T_vals))

    return optims[best_index], T_vals[best_index], solver[best_index]



##### Example usage

In [ ]:
n = 4 #Number of individuals
N_cell_each = 10000 #Number of cells each individual can occupy
print(HOMO_INTEGER_OPTIMISE_best(n, N_cell_each))

#### With foraging constraint

##### Optimisation 

In [ ]:

def HOMO_INTEGER_OPTIMISE_FORAGING_gp(n, N_cell_each, F_min, F_max, num_Fs):
    #Optimise in the homogeneous case with n individuals with N_cell_each knowledge capacities. With varying foraging constraint. Using Gurobi solver.
    with gp.Env(empty=True) as env: #To avoid text showing optimisation procedure. 
        env.setParam('OutputFlag', 0)
        env.start()

        #Pre-allocate space and define ranges
        individual_costs = [None] * num_Fs
        optims = [None] * num_Fs
        w_vals = np.zeros([num_Fs, n-1])
        T_vals = [None] * num_Fs
        
        Fs = np.linspace(F_min, F_max, num_Fs)
        Fs = np.ceil(Fs) #Make integers.
        #Setup independent to m
        X, N, X_sizes_increasing, X_sizes_decreasing = find_subsets(n)
        
        N_diff = (2**(n-1)) - 1
        entries = np.ones(n*N_diff + N)
        entries[n*N_diff:] = [-(len(X[i]) - 1) for i in range(N)]
        
        row_indices = np.zeros(n * N_diff+N)
        col_indices = [None] * (n * N_diff + N)
        for i in range(n):
            col_indices[i*N_diff:(i+1)*N_diff] = [X.index(set_) for set_ in X if {i}.issubset(set_)]
            row_indices[i*N_diff:(i+1)*N_diff] = i 
    
        col_indices[n*N_diff:] = range(N)
    
        row_indices[n*N_diff:] = n
            
        col_indices = np.array(col_indices)
        
        G = sp.csr_matrix((entries, (row_indices, col_indices)), shape=(n+1, N))

        optimal_sharing = np.inf
        
        #Set-up dependent upon F
        for i in reversed(range(num_Fs)):
            F = Fs[i]
            if optimal_sharing <= F:   #Old solution should still hold, so record previous values
                T_vals[i] = m.ObjVal
                optims[-1] = optim_Os
                for j in range(2, n+1):
                    w_vals[i, j-2] += get_w_i_n(X, optim_Os, N_cells, N, j, n)
                    
                individual_costs[i] = get_shared_and_unique(optim_Os, n, N_cells, X)[0][0]

            else: #optimise, as the old solution is now infeasible
                
                #Initialise the model
                # m = sci.Model()
                m = gp.Model(env = env, name = "test")
                
                #Define the variables (the overlapping points)
                O = m.addMVar(shape=N, vtype=GRB.INTEGER, name="O") #Constraining the solutions to be integers
                
        
                #Adding constraints
                m.addConstrs((O[i] >= 0 for i in range(N)), name = 'non-neg bounds') #For non-negativity
                
                N_cells = N_cell_each*np.ones(n)
                
                N_cells = [min(F, N_cells[i]) for i in range(n)]
        
                h = np.array(N_cells)
                h = np.append(h, [F-sum(N_cells)])
    
                # print(F-sum(N_cells))
                
                m.addConstr(G @ O <= h, name='sharing') #Defining the "don't share too much" constraint
    
        
                #Defining the objective function
                S_sets= [set(S(c, X, X_sizes_decreasing, N)) for c in range(N)]
                B_sets = [set(B(c, X, X_sizes_increasing, N)) for c in range(N)]
                sum_Ns, prod_Ns = give_sums_and_prods(X, N, N_cells)
                l = get_l(N, S_sets, sum_Ns, prod_Ns)
                M = get_M(X, N, S_sets, B_sets, prod_Ns)
                
                m.setObjective(l@O + 0.5 * O.T @ M @ O, GRB.MAXIMIZE)
                
                m.optimize()
                optim_Os = O.X
                optims[i] = optim_Os            
                T_vals[i] = m.ObjVal
                individual_costs[i] = get_shared_and_unique(optim_Os, n, N_cells, X)[0][0]
                
                for j in range(2, n+1):
                    w_vals[i, j-2] += get_w_i_n(X, optim_Os, N_cells, N, j, n)

                optimal_sharing = sum(N_cells)-sum([(len(X[f]) - 1)*optim_Os[f] for f in range(N)]) #To know if we need to iterate more

    return w_vals, optims, individual_costs, T_vals

def HOMO_INTEGER_OPTIMISE_FORAGING_xp(n, N_cell_each, F_min, F_max, num_Fs):
    #Optimise in the homogeneous case with n individuals with N_cell_each knowledge capacities. With varying foraging constraint. Using Xpress solver.
    xp.controls.outputlog = 0

    #Pre-allocate space and define ranges
    individual_costs = [None] * num_Fs
    optims = [None] * num_Fs
    w_vals = np.zeros([num_Fs, n-1])
    T_vals = [None] * num_Fs
    
    Fs = np.linspace(F_min, F_max, num_Fs)
    Fs = np.ceil(Fs) #Make integers.

    #Setup independent to m
    X, N, X_sizes_increasing, X_sizes_decreasing = find_subsets(n)
    index_sets = [[X.index(set_) for set_ in X if {i}.issubset(set_)] for i in range(n)]
    
    optimal_sharing = np.inf
    
    #Set-up dependent upon F
    for i in reversed(range(num_Fs)):
        F = Fs[i]
        if False:#optimal_sharing <= F:   #Old solution should still hold, so record previous values
            T_vals[i] = T_vals[i+1]
            optims[-1] = optim_Os
            for j in range(2, n+1):
                w_vals[i, j-2] += get_w_i_n(X, optim_Os, N_cells, N, j, n)
                
            individual_costs[i] = get_shared_and_unique(optim_Os, n, N_cells, X)[0][0]

        else: #optimise, as the old solution is now infeasible
            
            #Initialise the model
            m = xp.problem()
            
            #Define the variables (the overlapping points)
            X, N, X_sizes_increasing, X_sizes_decreasing = find_subsets(n)

            O = np.array([m.addVariable(lb=0, vartype = xp.integer, name = f'O_{i+1}') for i in range(N)])
            
            #We define the points, and then the environment determines the geography - that's the argument
            #Define the objective 
            N_cells = N_cell_each*np.ones(n)
            N_cells = [min(F, N_cells[i]) for i in range(n)]
            S_sets= [set(S(c, X, X_sizes_decreasing, N)) for c in range(N)]
            B_sets = [set(B(c, X, X_sizes_increasing, N)) for c in range(N)]
            sum_Ns, prod_Ns = give_sums_and_prods(X, N, N_cells)
        
        
            l = get_l(N, S_sets, sum_Ns, prod_Ns)
            M = get_M(X, N, S_sets, B_sets, prod_Ns)
            
            m.setObjective(l@O + 0.5 * O.T @ M @ O, sense = xp.maximize)

            
            #Adding constraints

            for a in range(n):
                m.addConstraint(xp.Sum(O[index_sets[a]]) <= N_cells[a])

            #Adding the F constraint
            m.addConstraint(sum(N_cells) - np.sum([(len(X[i]) - 1)*O[i] for i in range(N)]) <= F)   
    
            m.optimize()

    
            optim_Os = np.array(m.getSolution())
            optims[i] = optim_Os  
            T_vals[i] = l@optim_Os + 0.5 * optim_Os.T @ M @ optim_Os
            individual_costs[i] = get_shared_and_unique(optim_Os, n, N_cells, X)[0][0]
            
            for j in range(2, n+1):
                w_vals[i, j-2] += get_w_i_n(X, list(optim_Os), N_cells, N, j, n)

            #optimal_sharing = sum(N_cells)-sum([(len(X[f]) - 1)*optim_Os[f] for f in range(N)]) #To know if we need to iterate more

    return w_vals, optims, individual_costs, T_vals

def HOMO_INTEGER_OPTIMISE_FORAGING_cplex(n, N_cell_each, F_min, F_max, num_Fs):
#Optimise in the homogeneous case with n individuals with N_cell_each knowledge capacities. With varying foraging constraint. Using CPLEX solver.
    #Pre-allocate space and define ranges
    individual_costs = [None] * num_Fs
    optims = [None] * num_Fs
    w_vals = np.zeros([num_Fs, n-1])
    T_vals = [None] * num_Fs
    
    Fs = np.linspace(F_min, F_max, num_Fs)
    Fs = np.ceil(Fs) #Make integers.

    #Setup independent to m
    X, N, X_sizes_increasing, X_sizes_decreasing = find_subsets(n)
    S_sets= [set(S(c, X, X_sizes_decreasing, N)) for c in range(N)]
    B_sets = [set(B(c, X, X_sizes_increasing, N)) for c in range(N)]

    index_sets = [[X.index(set_) for set_ in X if {i}.issubset(set_)] for i in range(n)]
    
    optimal_sharing = np.inf
    
    #Set-up dependent upon F
    for i in reversed(range(num_Fs)):
        F = Fs[i]
        if False:#optimal_sharing <= F:   #Old solution should still hold, so record previous values
            T_vals[i] = T_vals[i+1]
            optims[-1] = optim_Os
            for j in range(2, n+1):
                w_vals[i, j-2] += get_w_i_n(X, optim_Os, N_cells, N, j, n)
                
            individual_costs[i] = get_shared_and_unique(optim_Os, n, N_cells, X)[0][0]

        else: #optimise, as the old solution is now infeasible
            
            #Initialise the model
            m = cplex.Cplex() 
            m.set_problem_type(m.problem_type.MIQP)
            m.objective.set_sense(m.objective.sense.maximize)
            
            
            #Define the variables and the objective 
            N_cells = N_cell_each*np.ones(n)
            N_cells = [min(F, N_cells[i]) for i in range(n)]
    
            sum_Ns, prod_Ns = give_sums_and_prods(X, N, N_cells)
        
        
            l = get_l(N, S_sets, sum_Ns, prod_Ns)
            M = get_M(X, N, S_sets, B_sets, prod_Ns)

            
            O = m.variables.add(obj=l, types=[m.variables.type.integer] * N, lb = [0] * N)
            m.objective.set_quadratic([cplex.SparsePair(ind = range(N), val = [M[i, k] for k in range(N)]) for i in range(N)])


            #Adding constraints
            
            m.linear_constraints.add(
                lin_expr = [cplex.SparsePair(ind = index_sets[i], val = [1] * len(index_sets[i])) for i in range(n)], 
            senses = ["L"] * n,
            rhs = N_cells) #Defining the "don't share too much" constraint

            #Adding the F constraint
            m.linear_constraints.add(lin_expr = [cplex.SparsePair(ind = range(N), val = [-(len(X[i]) - 1) for i in range(N)])],
            senses = ["L"],
            rhs = [F - sum(N_cells)])

            #Optimising
            m.parameters.optimalitytarget.set(3)

            #Hiding all of the output
            m.set_results_stream(None)
            m.set_log_stream(None)

            #Optimising
            m.parameters.timelimit.set(180) #3 min max 
            m.solve()   

            if m.solution.get_status() !=  119: #This is the status code for infeasible solution. Occasionally the solver fails due to its pre-processing methods.
                optim_Os = np.array(m.solution.get_values())
                optim_T = m.solution.get_objective_value()
            else: 
                optim_Os = np.zeros(N) #Assigning a global minima instead. Other solvers will then overrule this one. 
                optim_T = -1 #Assigning a value which will NOT be selected instead of the other solvers.           

    
            optim_Os = np.array(m.solution.get_values())
            optims[i] = optim_Os  
            T_vals[i] = optim_T
            individual_costs[i] = get_shared_and_unique(optim_Os, n, N_cells, X)[0][0]
            
            for j in range(2, n+1):
                w_vals[i, j-2] += get_w_i_n(X, list(optim_Os), N_cells, N, j, n)


    return w_vals, optims, individual_costs, T_vals

def HOMO_INTEGER_OPTIMISE_FORAGING_best(n, N_cell_each, F_min, F_max, num_Fs):
    #Optimise with each solver and return the best solution for each tested value of F
    individual_costs = [None] * num_Fs
    optims = [None] * num_Fs
    ws = np.zeros([num_Fs, n-1])
    Ts = [None] * num_Fs

    T_vals_all = [None] * 3
    ws_all = [None] * 3
    optims_all = [None] * 3
    individual_costs_all = [None] * 3

    print("With Gurobi...")
    ws_all[0], optims_all[0], individual_costs_all[0], T_vals_all[0] = HOMO_INTEGER_OPTIMISE_FORAGING_gp(n, N_cell_each, F_min, F_max, num_Fs) 
    print("With Xpress...")
    ws_all[1], optims_all[1], individual_costs_all[1], T_vals_all[1] = HOMO_INTEGER_OPTIMISE_FORAGING_xp(n, N_cell_each, F_min, F_max, num_Fs) 
    print("With CPLEX...")
    ws_all[2], optims_all[2], individual_costs_all[2], T_vals_all[2] = HOMO_INTEGER_OPTIMISE_FORAGING_cplex(n, N_cell_each, F_min, F_max, num_Fs) 

    for i in range(num_Fs):
        T_vals = [T_vals_all[0][i], T_vals_all[1][i], T_vals_all[2][i]]
        Ts[i] = max(T_vals)
        best_index = T_vals.index(Ts[i])

        individual_costs[i] = individual_costs_all[best_index][i]
        ws[i, :] = ws_all[best_index][i, :]
        optims[i] = optims_all[best_index][i]
    
    return ws, optims, individual_costs, Ts

##### Functions for plotting

In [ ]:

def plots_homo_constraints(n, ws, individual_costs, T_vals, F_min, F_max, num_Fs):
    #For plotting the w_n^(i) values, C_k values and the objective function in the optimal solution. 
    Fs = np.linspace(F_min, F_max, num_Fs)
    fig, ax = plt.subplots(1, n+1)
    for i in range(2, n+1):
        ax[i-2].plot(Fs, ws[:, i-2] , '.')
        ax[i-2].hlines(ws[-1, i-2], xmin=F_min, xmax=F_max, color='k', linestyles='--', label = "Value in case of unlimited resource")
        ax[i-2].set_xlim((F_min, F_max))
        ax[i-2].set_ylim((0-0.05, 1+0.05))
        ax[i-2].legend()
        ax[i-2].set_ylabel(f"Order {i} relative overlap, $w_{n}^{(i)}$")
        ax[i-2].set_xlabel(f"Available resource level, $F$")


    ax[n-1].plot(Fs, individual_costs, '.')
    ax[n-1].hlines(individual_costs[-1], xmin=F_min, xmax=F_max, color='k', linestyles='--', label = "Value in case of unlimited resource")
    ax[n-1].set_xlim((F_min, F_max))
    ax[n-1].legend()
    ax[n-1].set_ylabel(f"Individual-level costs")
    ax[n-1].set_xlabel(f"Available resource level, $F$")
        
    ax[n].plot(Fs, T_vals, 'r.')
    ax[n].hlines(T_vals[-1], xmin=F_min, xmax=F_max, color='k', linestyles='--', label = "Value in case of unlimited resource")
    ax[n].set_xlabel(f"Available resource level, $F$")
    ax[n].set_ylabel(f"Group-level average information transfer, $T$")
    ax[n].set_xlim((F_min, F_max))
    ax[n].legend()

    fig.set_figwidth(7*(n+1))
    fig.set_figheight(7)
        
    return fig, ax #Return the fig and ax themselves
        
def plot_w_means_homo_foraging(n, ws, F_min, F_max, num_Fs):
    #For plotting the value of K in the optimal spatial structure for varying F
    Fs = np.linspace(F_min, F_max, num_Fs)
    w_means = np.zeros(num_Fs)
    for i in range(num_Fs):
        w_means[i] = get_w_weight_mean(n, ws[i, :])
    fig = plt.plot(Fs, w_means, 'g.')
    plt.xlabel("Available resource level, $F$")
    plt.ylabel("Weighted mean $w$ value")
    return fig

##### Running model

In [ ]:
n = 5 #Number of individuals
N_cell_each = 10000 #Number of cells each individual can occupy
F_min = 7500 #Lowest number of maximum foraging points considered 
F_max = 25000 #Highest number of maximum foraging points considered 
num_Fs = 1000 #int((F_max-F_min+1)/20+1) #Number of different foraging constraints considered

ws, optims, individual_costs, Ts = HOMO_INTEGER_OPTIMISE_FORAGING_best(n, N_cell_each, F_min, F_max, num_Fs)#HOMO_INTEGER_OPTIMISE_FORAGING_best(n, N_cell_each, F_min, F_max, num_Fs)

fig, ax = plots_homo_constraints(n, ws, individual_costs, Ts, F_min, F_max, num_Fs)
plt.show()

fig = plot_w_means_homo_foraging(n, ws, F_min, F_max, num_Fs)
plt.show()

##### Some results

In [ ]:
fig, ax = plt.subplots(1,2)
Fs = np.linspace(F_min, F_max, num_Fs)

ftsz = 24

ws, optims, individual_costs, Ts = HOMO_INTEGER_OPTIMISE_FORAGING_best(3, N_cell_each, F_min, F_max, num_Fs)#HOMO_INTEGER_OPTIMISE_FORAGING_best(n, N_cell_each, F_min, F_max, num_Fs)

ax[0].plot(Fs, ws[:, 0], '-', linewidth = 4)
ax[0].set_ylabel(f'$w_3^2$', fontsize = ftsz)
ax[0].tick_params(axis='both', which='major', labelsize=14)
ax[0].tick_params(axis='both', which='minor', labelsize=14)
ax[0].vlines(20000, -0.025, 1, color='k', linestyles='--')
ax[0].vlines(10000, -0.025, 1, color='k', linestyles='--')
ax[0].plot([10000, 14997.4974975, 20000],[0, 1, 0], '.k', markersize = 20)
ax[0].set_ylim(-0.025, 1+0.025)
ax[0].set_xlim(F_min, F_max)
#ax[0].hlines(1, 7500, 14997.4974975, color='k', linestyles='--')



ax[1].plot(Fs, ws[:, 1], '-', linewidth = 4)
ax[1].set_ylabel(f'$w_3^3$', fontsize = ftsz)
ax[1].tick_params(axis='both', which='major', labelsize=14)
ax[1].tick_params(axis='both', which='minor', labelsize=14)
ax[1].vlines(20000, -0.025, 1, color='k', linestyles='--')
ax[1].vlines(10000, -0.025, 1, color='k', linestyles='--')
ax[1].plot([10000, 14997.4974975, 20000],[1, 0, 1/4], '.k', markersize = 20)
ax[1].set_ylim(-0.025, 1+0.025)
ax[1].set_xlim(F_min, F_max)


fig.set_figheight(7)
fig.set_figwidth(28)

plt.show()

#fig.savefig('ws_n3_lines.pdf', bbox_inches='tight')


In [ ]:
#Plotting for different n values.
N_cell_each = 10000
F_min = 500 #Lowest number of maximum foraging points considered 
F_max = 40000 #Highest number of maximum foraging points considered 
num_Fs = 4000 
lw = 4 #Linewidth

fig, ax = plt.subplots(1,3)

Fs = np.linspace(F_min, F_max, num_Fs)

ftsz = 24

plot_colours = ['d9ebcd', 'a3d6b3', '3fb6c4', '2c7fb9', '293990']

for n in range(3, 8):
    print(f"\nn={n}:")
    if n != 7:
        ws, optims, individual_costs, Ts = HOMO_INTEGER_OPTIMISE_FORAGING_best(n, N_cell_each, F_min, F_max, num_Fs)
    else: 
        ws, optims, individual_costs, Ts = HOMO_INTEGER_OPTIMISE_FORAGING_gp(n, N_cell_each, F_min, F_max, num_Fs)

    #Plotting T    
    ax[0].plot(Fs, Ts, '-', label = f"$n=${n}", linewidth = lw, color = '#' + plot_colours[n-3])
    ax[0].set_xlabel(f"Available resource level, $F$", fontsize = ftsz)
    ax[0].set_ylabel(f"Group information transfer, $T$", fontsize = ftsz)
    
    w_means = np.zeros(num_Fs)
    for i in range(num_Fs):
        w_means[i] = get_w_weight_mean(n, ws[i, :])
    ax[1].plot(Fs, w_means, '-', label = f"$n=${n}", linewidth = lw, color = '#' + plot_colours[n-3])
    ax[1].set_xlabel("Available resource level, $F$", fontsize = ftsz)
    ax[1].set_ylabel("Average point knowledge, $K$", fontsize = ftsz)

    if n!= 5:
        ax[2].plot(Fs, individual_costs, '-', label = f"$n=${n}", linewidth = lw, color = '#' + plot_colours[n-3])
    else:
        ax[2].plot(Fs, individual_costs, '.', label = f"$n=${n}", color = '#' + plot_colours[n-3])
    ax[2].set_ylabel(f"Individual sharing, $C_k$", fontsize = ftsz)
    ax[2].set_xlabel(f"Available resource level, $F$", fontsize = ftsz)

for i in range(3):
    ax[i].tick_params(axis='both', which='major', labelsize=14)

fig.set_figwidth(28)
fig.set_figheight(7)

plt.show()




In [ ]:
#Trying to understand why there is the incosnistency in the n=5 individual cost plot.
n = 5
N_cell_each = 10000
F_min = 20000
F_max = 30000
num_Fs = 200
Fs = np.linspace(F_min, F_max, num_Fs)
ws, optims, individual_costs, Ts = HOMO_INTEGER_OPTIMISE_FORAGING_best(n, N_cell_each, F_min, F_max, num_Fs)

fig, ax = plt.subplots()

ax.plot(Fs, individual_costs)

plt.show()

## Scenario 2: Distinct forager

#### Without foraging constraint

##### Optimisation

In [ ]:
#Unique forager, Gurobi
def ONE_DIFF_INTEGER_OPTIMISE_gp(n, N_cell_each, D_min, D_max, num_Ds):
    #Optimise in the case with n individuals with N_cell_each knowledge capacities, except individual 1 with D times this ability. 
    # No foraging constraint. Compute for range of D values. Using Gurobi solver.
    with gp.Env(empty=True) as env: #To avoid text showing optimisation procedure. 
        env.setParam('OutputFlag', 0)
        env.start()

        #Pre-allocate space and define ranges
        optims = [None] * num_Ds
        w_vals = np.zeros([num_Ds, n-1])
        T_vals = [None] * num_Ds
        individual_costs = [None] * num_Ds
        
        
        Ds = np.linspace(D_min, D_max, num_Ds)

        
    
        #Setup independent to m
        X, N, X_sizes_increasing, X_sizes_decreasing = find_subsets(n)
        
        N_diff = (2**(n-1)) - 1
        entries = np.ones(n*N_diff)
        
        row_indices = np.zeros(n * N_diff)
        col_indices = [None] * (n * N_diff)
        for i in range(n):
            col_indices[i*N_diff:(i+1)*N_diff] = [X.index(set_) for set_ in X if {i}.issubset(set_)]
            row_indices[i*N_diff:(i+1)*N_diff] = i 
            
        col_indices = np.array(col_indices)
    
        G = sp.csr_matrix((entries, (row_indices, col_indices)), shape=(n, N))
    
    
        #setup involving m, and optimisation
        for i in range(num_Ds):
            D = Ds[i] #Current individual scaling
    
            #Initialise the model
            m = gp.Model(env = env, name = "test")
            
            #Define the variables (the overlapping points)
            O = m.addMVar(shape=N, vtype=GRB.INTEGER, name="O") #Constraining the solutions to be integers
    
            #Adding constraints
            m.addConstrs((O[i] >= 0 for i in range(N)), name = 'non-neg bounds') #For non-negativity
            
            N_cells = N_cell_each*np.ones(n)
            N_cells[0] = int(np.round(D * N_cells[0])) #Arbitrarily picking the first individual to be the 'different' forager
                
            h = np.array(N_cells)
            
            m.addConstr(G @ O <= h, name='sharing') #Defining the "don't share too much" constraint
    
            #Defining the objective function
            S_sets= [set(S(c, X, X_sizes_decreasing, N)) for c in range(N)]
            B_sets = [set(B(c, X, X_sizes_increasing, N)) for c in range(N)]
            sum_Ns, prod_Ns = give_sums_and_prods(X, N, N_cells)
            l = get_l(N, S_sets, sum_Ns, prod_Ns)
            M = get_M(X, N, S_sets, B_sets, prod_Ns)
            
            m.setObjective(l@O + 0.5 * O.T @ M @ O, GRB.MAXIMIZE)

            #Optimising 
            m.optimize()

            #Storing data
            optim_Os = O.X
            optims[i] = optim_Os
            T_vals[i] = m.ObjVal
            individual_costs[i] = get_shared_and_unique(optim_Os, n, N_cells, X)[0]
                
            for j in range(2, n+1):
                w_vals[i, j-2] += get_w_i_n(X, optim_Os, N_cells, N, j, n)


    costs_different = [individual_costs[i][0] for i in range(num_Ds)]
    costs_normal = [individual_costs[i][1] for i in range(num_Ds)]

    return w_vals, optims, costs_different, costs_normal, T_vals

#Unique forager, FICO Xpress
def ONE_DIFF_INTEGER_OPTIMISE_xp(n, N_cell_each, D_min, D_max, num_Ds):
    #Optimise in the case with n individuals with N_cell_each knowledge capacities, except individual 1 with D times this ability. 
    # No foraging constraint. Compute for range of D values. Using Xpress solver.
    xp.controls.outputlog = 0

    #Pre-allocate space and define ranges
    optims = [None] * num_Ds
    w_vals = np.zeros([num_Ds, n-1])
    T_vals = [None] * num_Ds
    individual_costs = [None] * num_Ds
    
    
    Ds = np.linspace(D_min, D_max, num_Ds)

    #Setup independent to m
    X, N, X_sizes_increasing, X_sizes_decreasing = find_subsets(n)
    index_sets = [[X.index(set_) for set_ in X if {i}.issubset(set_)] for i in range(n)]
  

    #setup involving m, and optimisation
    for i in range(num_Ds):
        D = Ds[i] #Current individual scaling

        #Initialise the model
        m = xp.problem()
        
        #Define the variables (the overlapping points)
        O = np.array([m.addVariable(lb=0, vartype = xp.integer, name = f'O_{i+1}') for i in range(N)]) #Constraining the solutions to be integers

        #Defining the objective
        N_cells = N_cell_each*np.ones(n)
        N_cells[0] = int(np.round(D * N_cells[0])) #Arbitrarily picking the first individual to be the 'different' forager

        S_sets= [set(S(c, X, X_sizes_decreasing, N)) for c in range(N)]
        B_sets = [set(B(c, X, X_sizes_increasing, N)) for c in range(N)]
        sum_Ns, prod_Ns = give_sums_and_prods(X, N, N_cells)

        l = get_l(N, S_sets, sum_Ns, prod_Ns)
        M = get_M(X, N, S_sets, B_sets, prod_Ns)
            
        m.setObjective(l@O + 0.5 * O.T @ M @ O, sense = xp.maximize)

        #Adding constraints

        for a in range(n):
            m.addConstraint(xp.Sum(O[index_sets[a]]) <= N_cells[a])
           
        #Optimising 
        m.optimize()

        #Storing data
        optim_Os = np.array(m.getSolution())
        optims[i] = optim_Os  
        T_vals[i] = l@optim_Os + 0.5 * optim_Os.T @ M @ optim_Os
        individual_costs[i] = get_shared_and_unique(optim_Os, n, N_cells, X)[0]
            
        for j in range(2, n+1):
            w_vals[i, j-2] += get_w_i_n(X, list(optim_Os), N_cells, N, j, n)


    costs_different = [individual_costs[i][0] for i in range(num_Ds)]
    costs_normal = [individual_costs[i][1] for i in range(num_Ds)]

    return w_vals, optims, costs_different, costs_normal, T_vals

#Unique forager, CPLEX
def ONE_DIFF_INTEGER_OPTIMISE_cplex(n, N_cell_each, D_min, D_max, num_Ds):
# Optimise in the case with n individuals with N_cell_each knowledge capacities, except individual 1 with D times this ability. 
# No foraging constraint. Compute for range of D values. Using CPLEX solver.   

    #Pre-allocate space and define ranges
    optims = [None] * num_Ds
    w_vals = np.zeros([num_Ds, n-1])
    T_vals = [None] * num_Ds
    individual_costs = [None] * num_Ds
    
    
    Ds = np.linspace(D_min, D_max, num_Ds)

    #Setup independent to m
    X, N, X_sizes_increasing, X_sizes_decreasing = find_subsets(n)
    S_sets= [set(S(c, X, X_sizes_decreasing, N)) for c in range(N)]
    B_sets = [set(B(c, X, X_sizes_increasing, N)) for c in range(N)]
    index_sets = [[X.index(set_) for set_ in X if {i}.issubset(set_)] for i in range(n)]
  

    #setup involving m, and optimisation
    for i in range(num_Ds):
        D = Ds[i] #Current individual scaling

        #Initialise the model
        m = cplex.Cplex() 
        m.set_problem_type(m.problem_type.MIQP)
        m.objective.set_sense(m.objective.sense.maximize)
        
        #Define the variables and the objective 
        N_cells = N_cell_each*np.ones(n)
        N_cells[0] = int(np.round(D * N_cells[0])) #Arbitrarily picking the first individual to be the 'different' forager

        sum_Ns, prod_Ns = give_sums_and_prods(X, N, N_cells)

        l = get_l(N, S_sets, sum_Ns, prod_Ns)
        M = get_M(X, N, S_sets, B_sets, prod_Ns)
            
        O = m.variables.add(obj=l, types=[m.variables.type.integer] * N, lb = [0] * N)
        m.objective.set_quadratic([cplex.SparsePair(ind = range(N), val = [M[i, k] for k in range(N)]) for i in range(N)])

        #Adding constraints

        m.linear_constraints.add(
            lin_expr = [cplex.SparsePair(ind = index_sets[i], val = [1] * len(index_sets[i])) for i in range(n)], 
            senses = ["L"] * n,
            rhs = N_cells) #Defining the "don't share too much" constraint
           
        #Optimising
        m.parameters.optimalitytarget.set(3)

        #Hiding all of the output
        m.set_results_stream(None)
        m.set_log_stream(None)

        m.solve()       

        #Storing data
        optim_Os = np.array(m.solution.get_values())
        optims[i] = optim_Os  
        T_vals[i] = m.solution.get_objective_value()
        individual_costs[i] = get_shared_and_unique(optim_Os, n, N_cells, X)[0]
            
        for j in range(2, n+1):
            w_vals[i, j-2] += get_w_i_n(X, list(optim_Os), N_cells, N, j, n)


    costs_different = [individual_costs[i][0] for i in range(num_Ds)]
    costs_normal = [individual_costs[i][1] for i in range(num_Ds)]

    return w_vals, optims, costs_different, costs_normal, T_vals

#Using three above functions to get the best solution
def ONE_DIFF_INTEGER_OPTIMISE_best(n, N_cell_each, D_min, D_max, num_Ds):
    #Optimise with each solver and return the best solution for each tested value of D
    costs_different = [None] * num_Ds
    costs_normal = [None] * num_Ds
    optims = [None] * num_Ds
    ws = np.zeros([num_Ds, n-1])
    Ts = [None] * num_Ds

    T_vals_all = [None] * 3
    ws_all = [None] * 3
    optims_all = [None] * 3
    costs_different_all = [None] * 3
    costs_normal_all = [None] * 3

    print("With Gurobi...")
    ws_all[0], optims_all[0], costs_different_all[0], costs_normal_all[0], T_vals_all[0] = ONE_DIFF_INTEGER_OPTIMISE_gp(n, N_cell_each, D_min, D_max, num_Ds) 
    print("With Xpress...")
    ws_all[1], optims_all[1], costs_different_all[1], costs_normal_all[1], T_vals_all[1] = ONE_DIFF_INTEGER_OPTIMISE_xp(n, N_cell_each, D_min, D_max, num_Ds) 
    print("With CPLEX...")
    ws_all[2], optims_all[2], costs_different_all[2], costs_normal_all[2], T_vals_all[2] = ONE_DIFF_INTEGER_OPTIMISE_cplex(n, N_cell_each, D_min, D_max, num_Ds) 

    for i in range(num_Ds):
        T_vals = [T_vals_all[0][i], T_vals_all[1][i], T_vals_all[2][i]]
        Ts[i] = max(T_vals)
        best_index = T_vals.index(Ts[i])

        costs_different[i] = costs_different_all[best_index][i]
        costs_normal[i] = costs_normal_all[best_index][i]
        ws[i, :] = ws_all[best_index][i, :]
        optims[i] = optims_all[best_index][i]

    
    return ws, optims, costs_different, costs_normal, Ts


##### Functions for plotting

In [ ]:
#Function for summary plotting of results
def plots_one_diff(n, w_vals, costs_different, costs_normal, T_vals, D_min, D_max, num_Ds):
    #For plotting the w_n^(i) values, C_k values and the objective function in the optimal solution. 
    Ds = np.linspace(D_min, D_max, num_Ds)
    fig, ax = plt.subplots(1, n+2)
    for i in range(2, n+1):
        ax[i-2].plot(Ds, w_vals[:, i-2] , '.')
        ax[i-2].set_xlim((D_min, D_max))
        ax[i-2].set_ylabel(f"Order {i} relative overlap, $w_{n}^{(i)}$")
        ax[i-2].set_xlabel(f"Scale difference in one individual, $D$")
        ax[i-2].set_ylim((0-0.05, 1+0.05))

    ax[n-1].plot(Ds, costs_different, '.')
    ax[n-1].set_xlim((D_min, D_max))
    ax[n-1].set_ylabel(f"Individual-level costs, different ability")
    ax[n-1].set_xlabel(f"Scale difference in one individual, $D$")
    ax[n-1].set_ylim((0-0.05, 1+0.05))
    
    ax[n].plot(Ds, costs_normal, '.')
    ax[n].set_xlim((D_min, D_max))
    ax[n].set_ylabel(f"Individual-level costs, regular ability")
    ax[n].set_xlabel(f"Scale difference in one individual, $D$")
    ax[n].set_ylim((0-0.05, 1+0.05))
    
    ax[n+1].plot(Ds, T_vals, 'r.')
    ax[n+1].set_xlabel(f"Scale difference in one individual, $D$")
    ax[n+1].set_ylabel(f"Group-level average information transfer, $T$")
    ax[n+1].set_xlim((D_min, D_max))
     
    fig.set_figwidth(7*(n+2))
    fig.set_figheight(7)

    return fig, ax

def plot_w_means_one_diff(n, ws, D_min, D_max, num_Ds):
    #For plotting the value of K in the optimal spatial structure for varying D
    Ds = np.linspace(D_min, D_max, num_Ds)
    w_means = np.zeros(num_Ds)
    for i in range(num_Ds):
        w_means[i] = get_w_weight_mean(n, ws[i, :])
    fig = plt.plot(Ds, w_means, 'g.')
    plt.xlabel("Scale difference in one individual, $D$")
    plt.ylabel("Weighted mean $w$ value")
    return fig

##### Running model

In [ ]:
#Example usage

n = 4 #Number of individuals 
N_cell_each = 10000#Number of cells each individual can occupy
D_min = 1 #Scaling for worst forager considered
D_max = 10 #Scaling for best 'different forager' considered
num_Ds = 1000 #Number of different ability scalings considered


ws, optims, costs_different, costs_normal, Ts = ONE_DIFF_INTEGER_OPTIMISE_best(n, N_cell_each, D_min, D_max, num_Ds)

fig, ax = plots_one_diff(n, ws, costs_different, costs_normal, Ts, D_min, D_max, num_Ds)
plt.show()

fig = plot_w_means_one_diff(n, ws, D_min, D_max, num_Ds)
plt.show()

##### Some results

In [ ]:
N_cell_each = 10000#Number of cells each individual can occupy
D_min = 0.01 #Scaling for worst forager considered
D_max = 6 #Scaling for best 'different forager' considered
num_Ds = 1000 #Number of different ability scalings considered
ftsz = 20 #Font size
lw = 4 #Linewidth

fig, ax = plt.subplots(2,2)

fig.set_figwidth(28)
fig.set_figheight(7*1.55)

line_colors = ['#377eb8', '#ff7f00', '#4daf4a',
                  '#f781bf', '#a65628', '#984ea3',
                  '#999999', '#e41a1c', '#dede00']

Ds_worse = np.linspace(D_min, 1, num_Ds)
Ds_better = np.linspace(1, D_max, num_Ds)

for n in [3, 4, 5, 6, 7]:
    print(f'\nn={n}')
    #Worse first
    if n!=6 and n!=7:
        ws, optims, costs_different, costs_normal, Ts = ONE_DIFF_INTEGER_OPTIMISE_best(n, N_cell_each, D_min, 1, num_Ds)
    else: 
        ws, optims, costs_different, costs_normal, Ts = ONE_DIFF_INTEGER_OPTIMISE_gp(n, N_cell_each, D_min, 1, num_Ds)

    w_means = np.zeros(num_Ds)
    for i in range(num_Ds):
        w_means[i] = get_w_weight_mean(n, ws[i, :])

    ax[0, 0].plot(Ds_worse, Ts, '-', label = f'n={n}', color = line_colors[n-3], linewidth = lw)
    ax[0, 1].plot(Ds_worse, w_means, '-', label = f'n={n}', color = line_colors[n-3], linewidth = lw)
    ax[1, 0].plot(Ds_worse, costs_different, '-', label = f'n={n}', color = line_colors[n-3], linewidth = lw)
    ax[1, 1].plot(Ds_worse, costs_normal, '-', label = f'n={n}', color = line_colors[n-3], linewidth = lw)

    #Then better
    if n!=6 and n!=7:
        ws, optims, costs_different, costs_normal, Ts = ONE_DIFF_INTEGER_OPTIMISE_best(n, N_cell_each, 1, D_max, num_Ds)
    else: 
        ws, optims, costs_different, costs_normal, Ts = ONE_DIFF_INTEGER_OPTIMISE_gp(n, N_cell_each, 1, D_max, num_Ds)
    w_means = np.zeros(num_Ds)
    for i in range(num_Ds):
        w_means[i] = get_w_weight_mean(n, ws[i, :])

    ax[0, 0].plot(Ds_better, Ts, '-', color = line_colors[n-3], linewidth = lw)
    ax[0, 1].plot(Ds_better, w_means, '-', color = line_colors[n-3], linewidth = lw)
    ax[1, 0].plot(Ds_better, costs_different, '-', color = line_colors[n-3], linewidth = lw)
    ax[1, 1].plot(Ds_better, costs_normal, '-', color = line_colors[n-3], linewidth = lw)

    ax[0, 0].plot([1], Ts[0], '.k', markersize = 15)
    ax[0, 1].plot([1], w_means[0], '.k', markersize = 15)
    ax[1, 0].plot([1], costs_different[0], '.k', markersize = 15)
    ax[1, 1].plot([1], costs_normal[0], '.k', markersize = 15)

ax[0, 0].legend(fontsize = 16)
ax[0, 1].legend(fontsize = 16)
ax[1, 0].legend(fontsize = 16)
ax[1, 1].legend(fontsize = 16)

#ax[0, 0].set_xlabel("Scale difference in one individual, $D$", fontsize = ftsz)
#ax[0, 1].set_xlabel("Scale difference in one individual, $D$", fontsize = ftsz)
ax[1, 0].set_xlabel("Scale difference in one individual, $D$", fontsize = ftsz)
ax[1, 1].set_xlabel("Scale difference in one individual, $D$", fontsize = ftsz)

ax[0, 0].set_ylabel("Group information transfer, $T$", fontsize = ftsz)
ax[0, 1].set_ylabel("Average point knowledge, $K$", fontsize = ftsz)
ax[1, 0].set_ylabel("Unique forager sharing, $C_1$", fontsize = ftsz)
ax[1, 1].set_ylabel("Regular forager sharing, $C_k$", fontsize = ftsz)

ax[0, 0].tick_params(axis='both', which='major', labelsize=14)
ax[0, 1].tick_params(axis='both', which='major', labelsize=14)
ax[1, 0].tick_params(axis='both', which='major', labelsize=14)
ax[1, 1].tick_params(axis='both', which='major', labelsize=14)

ax[0, 0].set_xlim(D_min, D_max)
ax[0, 1].set_xlim(D_min, D_max)
ax[1, 0].set_xlim(D_min, D_max)
ax[1, 1].set_xlim(D_min, D_max)

ax[0,0].set_ylim(0.75, 14.25)
ax[0,1].set_ylim(1.25, 3.25)
ax[1,0].set_ylim(0-0.05, 1+0.05)
ax[1,1].set_ylim(0-0.05, 1+0.05)

lims = ax[0,0].get_ylim()
ax[0, 0].vlines(1, lims[0], lims[1], color='k', linestyles='--')

lims = ax[0,1].get_ylim()
ax[0, 1].vlines(1, lims[0], lims[1], color='k', linestyles='--')
lims = ax[1,0].get_ylim()
ax[1, 0].vlines(1, lims[0], lims[1], color='k', linestyles='--')
lims = ax[1,1].get_ylim()
ax[1, 1].vlines(1, lims[0], lims[1], color='k', linestyles='--')

ax[0,0].set_ylim(0.75, 14.25)
ax[0,1].set_ylim(1.25, 3.25)
ax[1,0].set_ylim(0-0.05, 1+0.05)
ax[1,1].set_ylim(0-0.05, 1+0.05)

plt.show()



In [ ]:
#Plot of the w's
n = 4
N_cell_each = 10000#Number of cells each individual can occupy
D_min = 0.01 #Scaling for worst forager considered
D_max = 6 #Scaling for best 'different forager' considered
num_Ds = 1000 #Number of different ability scalings considered

ftsz = 20

fig, ax = plt.subplots(3)

fig.set_figwidth(28)
fig.set_figheight(7*2.2)

line_colors = ['#377eb8', '#ff7f00', '#4daf4a',
                  '#f781bf', '#a65628', '#984ea3',
                  '#999999', '#e41a1c', '#dede00'] #From random online post

Ds_worse = np.linspace(D_min, 1, num_Ds)
Ds_better = np.linspace(1, D_max, num_Ds)

for n in [4,5,6]:
    print(f"\nn={n}")
    j = n-4
    #First worse
    if n!=6 and n!=7:
        ws, optims, costs_different, costs_normal, Ts = ONE_DIFF_INTEGER_OPTIMISE_best(n, N_cell_each, D_min, 1, num_Ds)
    else: 
        ws, optims, costs_different, costs_normal, Ts = ONE_DIFF_INTEGER_OPTIMISE_gp(n, N_cell_each, D_min, 1, num_Ds)
    for i in range(2, n+1):
        ax[j].plot(Ds_worse, ws[:, i-2], '-', label = f'i={i}', color = line_colors[i], linewidth = lw)
        ax[j].set_ylabel(f'$w_{n}^i$', fontsize = ftsz)
        
        

    #Then better
    if n!=6 and n!=7 and n!= 8:
        ws, optims, costs_different, costs_normal, Ts = ONE_DIFF_INTEGER_OPTIMISE_best(n, N_cell_each, 1, D_max, num_Ds)
    else: 
        ws, optims, costs_different, costs_normal, Ts = ONE_DIFF_INTEGER_OPTIMISE_gp(n, N_cell_each, 1, D_max, num_Ds)
    for i in range(2, n+1):
        ax[j].plot(Ds_better, ws[:, i-2], '-', color = line_colors[i], linewidth = lw)
        ax[j].legend(fontsize = 16)
        ax[j].tick_params(axis='both', which='major', labelsize=18)
        ax[j].tick_params(axis='both', which='minor', labelsize=18)
        ax[j].set_xlim(D_min, D_max)
        ax[j].vlines(1, 0-0.05, 0.6, color='k', linestyles='--')
        ax[j].plot([Ds_worse[0], 1], [1/n, ws[0, i-2]], '.k', markersize = 15)
    
    ax[j].set_ylim(0-0.05, 0.6)
    ax[j].set_xlim(D_min, D_max)

ax[-1].set_xlabel(f"Scale difference in one individual, $D$", fontsize = ftsz)

    #ax[j].plot([Ds_worse[0], 1], [1/n, ws[0, i-2]], '.k', markersize = 20)

#fig.savefig("Unique_ws.pdf", bbox_inches = "tight")
plt.show()


### With foraging constraint

##### Optimisation

In [ ]:
def ONE_DIFF_INTEGER_OPTIMISE_FORAGE_gp(n, N_cell_normal, D_min, D_max, num_Ds, F_min, F_max, num_Fs):
    # Optimise in the case with n individuals with N_cell_each knowledge capacities, except individual 1 with D times this ability. 
    # With varying foraging constraint. Compute for range of D values. Using Gurobi solver. 
    with gp.Env(empty=True) as env: #To avoid text showing optimisation procedure. 
        env.setParam('OutputFlag', 0)
        env.start()

        #Pre-allocate space and define ranges
        individual_costs = [[None] * num_Ds for i in range(num_Fs)]
        optims = [[None] * num_Ds for i in range(num_Fs)]
        w_vals = np.zeros([num_Fs, num_Ds, n-1])
        
        T_vals = np.zeros((num_Fs, num_Ds))
        
        Fs = np.linspace(F_min, F_max, num_Fs)
        Ds = np.linspace(D_min, D_max, num_Ds)

        Fs = np.floor(Fs)

    
        #Setup independent to m
        X, N, X_sizes_increasing, X_sizes_decreasing = find_subsets(n)
        
        N_diff = (2**(n-1)) - 1
        entries = np.ones(n*N_diff + N)
        entries[n*N_diff:] = [-(len(X[i]) - 1) for i in range(N)]
        
        row_indices = np.zeros(n * N_diff+N)
        col_indices = [None] * (n * N_diff + N)
        for i in range(n):
            col_indices[i*N_diff:(i+1)*N_diff] = [X.index(set_) for set_ in X if {i}.issubset(set_)]
            row_indices[i*N_diff:(i+1)*N_diff] = i 
    
        col_indices[n*N_diff:] = range(N)
    
        row_indices[n*N_diff:] = n
            
        col_indices = np.array(col_indices)
        
        G = sp.csr_matrix((entries, (row_indices, col_indices)), shape=(n+1, N))
    
        #setup involving m, and optimisation
        for i in range(num_Ds):
            D = Ds[i] #Current individual scaling
            #print(i)
            for j in range(num_Fs):
                F = Fs[j] #Current foraging constraint
                
                #Initialise the model
                m = gp.Model(env = env, name = "test")
                
                #Define the variables (the overlapping points)
                O = m.addMVar(shape=N, vtype=GRB.INTEGER, name="O") #Constraining the solutions to be integers
        
                #Adding constraints
                m.addConstrs((O[i] >= 0 for i in range(N)), name = 'non-neg bounds') #For non-negativity
                
                N_cells = N_cell_normal*np.ones(n)
                N_cells[0] = np.ceil(D * N_cells[0]) #Arbitrarily picking the first individual to be the 'different' forager
                N_cells = [min(F, N_cells[i]) for i in range(n)]
    
                h = np.array(N_cells)
                h = np.append(h, [F-sum(N_cells)])
                
                m.addConstr(G @ O <= h, name='sharing') #Defining the "don't share too much" constraint
        
                #Defining the objective function
                S_sets= [set(S(c, X, X_sizes_decreasing, N)) for c in range(N)]
                B_sets = [set(B(c, X, X_sizes_increasing, N)) for c in range(N)]
                sum_Ns, prod_Ns = give_sums_and_prods(X, N, N_cells)
                l = get_l(N, S_sets, sum_Ns, prod_Ns)
                M = get_M(X, N, S_sets, B_sets, prod_Ns)
                
                m.setObjective(l@O + 0.5 * O.T @ M @ O, GRB.MAXIMIZE)
    
                #Optimising 
                m.optimize()
    
                #Storing data
                optim_Os = O.X
                optims[j][i] = optim_Os #list indexing
                T_vals[j, i] = m.ObjVal #numpy array indexing
                individual_costs[j][i] = get_shared_and_unique(optim_Os, n, N_cells, X)[0]
                    
                for k in range(2, n+1):
                    w_vals[j, i, k-2] += get_w_i_n(X, optim_Os, N_cells, N, k, n)


    costs_different = [[None] * num_Ds for i in range(num_Fs)] 
    costs_normal = [[None] * num_Ds for i in range(num_Fs)]
    
    for i in range(num_Ds):
        for j in range(num_Fs):
            costs_different[j][i] = individual_costs[j][i][0]
            costs_normal[j][i] = individual_costs[j][i][1]

    return w_vals, optims, costs_different, costs_normal, T_vals

def ONE_DIFF_INTEGER_OPTIMISE_FORAGE_xp(n, N_cell_normal, D_min, D_max, num_Ds, F_min, F_max, num_Fs):
    # Optimise in the case with n individuals with N_cell_each knowledge capacities, except individual 1 with D times this ability. 
    # With varying foraging constraint. Compute for range of D values. Using Xpress solver. 
    xp.controls.outputlog = 0

    #Pre-allocate space and define ranges
    individual_costs = [[None] * num_Ds for i in range(num_Fs)]
    optims = [[None] * num_Ds for i in range(num_Fs)]
    w_vals = np.zeros([num_Fs, num_Ds, n-1])
    
    T_vals = np.zeros((num_Fs, num_Ds))
    
    Fs = np.linspace(F_min, F_max, num_Fs)
    Ds = np.linspace(D_min, D_max, num_Ds)

    Fs = np.floor(Fs)

    #Setup independent to m
    X, N, X_sizes_increasing, X_sizes_decreasing = find_subsets(n)
    index_sets = [[X.index(set_) for set_ in X if {i}.issubset(set_)] for i in range(n)]

    #setup involving m, and optimisation
    for i in range(num_Ds):
        D = Ds[i] #Current individual scaling
        for j in range(num_Fs):
            F = Fs[j] #Current foraging constraint
            
            #Initialise the model
            m = xp.problem()
            
            #Define the variables (the overlapping points)
            O = np.array([m.addVariable(lb=0, vartype = xp.integer, name = f'O_{i+1}') for i in range(N)]) #Constraining the solutions to be integers
            
            #Defining the objective
            N_cells = N_cell_normal*np.ones(n)
            N_cells[0] = int(np.round(D * N_cells[0])) #Arbitrarily picking the first individual to be the 'different' forager
            N_cells = [min(F, N_cells[i]) for i in range(n)]

            S_sets= [set(S(c, X, X_sizes_decreasing, N)) for c in range(N)]
            B_sets = [set(B(c, X, X_sizes_increasing, N)) for c in range(N)]
            sum_Ns, prod_Ns = give_sums_and_prods(X, N, N_cells)

            l = get_l(N, S_sets, sum_Ns, prod_Ns)
            M = get_M(X, N, S_sets, B_sets, prod_Ns)
                
            m.setObjective(l@O + 0.5 * O.T @ M @ O, sense = xp.maximize)

            #Adding constraints
            for a in range(n):
                m.addConstraint(xp.Sum(O[index_sets[a]]) <= N_cells[a])

            m.addConstraint(sum(N_cells) - np.sum([(len(X[i]) - 1)*O[i] for i in range(N)]) <= F)   

            #Optimising 
            m.optimize()

            #Storing data
            optim_Os = np.array(m.getSolution())
            optims[j][i] = optim_Os  
            T_vals[j, i] = l@optim_Os + 0.5 * optim_Os.T @ M @ optim_Os
            individual_costs[j][i] = get_shared_and_unique(optim_Os, n, N_cells, X)[0]
                
            for k in range(2, n+1):
                w_vals[j, i, k-2] += get_w_i_n(X, optim_Os, N_cells, N, k, n)

    costs_different = [[None] * num_Ds for i in range(num_Fs)] 
    costs_normal = [[None] * num_Ds for i in range(num_Fs)]
    
    for i in range(num_Ds):
        for j in range(num_Fs):
            costs_different[j][i] = individual_costs[j][i][0]
            costs_normal[j][i] = individual_costs[j][i][1]

    return w_vals, optims, costs_different, costs_normal, T_vals

def ONE_DIFF_INTEGER_OPTIMISE_FORAGE_cplex(n, N_cell_normal, D_min, D_max, num_Ds, F_min, F_max, num_Fs):
# Optimise in the case with n individuals with N_cell_each knowledge capacities, except individual 1 with D times this ability. 
# With varying foraging constraint. Compute for range of D values. Using CPLEX solver. 
    #Pre-allocate space and define ranges
    individual_costs = [[None] * num_Ds for i in range(num_Fs)]
    optims = [[None] * num_Ds for i in range(num_Fs)]
    w_vals = np.zeros([num_Fs, num_Ds, n-1])
    T_vals = np.zeros((num_Fs, num_Ds))
    
    Fs = np.linspace(F_min, F_max, num_Fs)
    Ds = np.linspace(D_min, D_max, num_Ds)

    Fs = np.floor(Fs)

    #Setup independent to m
    X, N, X_sizes_increasing, X_sizes_decreasing = find_subsets(n)
    S_sets= [set(S(c, X, X_sizes_decreasing, N)) for c in range(N)]
    B_sets = [set(B(c, X, X_sizes_increasing, N)) for c in range(N)]
    index_sets = [[X.index(set_) for set_ in X if {i}.issubset(set_)] for i in range(n)]

    #setup involving m, and optimisation
    for i in range(num_Ds):
        D = Ds[i] #Current individual scaling
        for j in range(num_Fs):
            F = Fs[j] #Current foraging constraint
            
            #Initialise the model
            m = cplex.Cplex() 
            m.set_problem_type(m.problem_type.MIQP)
            m.objective.set_sense(m.objective.sense.maximize)
            
            
            #Defining the objective and the variables
            N_cells = N_cell_normal*np.ones(n)
            N_cells[0] = int(np.round(D * N_cells[0])) #Arbitrarily picking the first individual to be the 'different' forager
            N_cells = [min(F, N_cells[i]) for i in range(n)]

            sum_Ns, prod_Ns = give_sums_and_prods(X, N, N_cells)

            l = get_l(N, S_sets, sum_Ns, prod_Ns)
            M = get_M(X, N, S_sets, B_sets, prod_Ns)
                
            O = m.variables.add(obj=l, types=[m.variables.type.integer] * N, lb = [0] * N)
            m.objective.set_quadratic([cplex.SparsePair(ind = range(N), val = [M[i, k] for k in range(N)]) for i in range(N)])

            #Adding constraints
            m.linear_constraints.add(
            lin_expr = [cplex.SparsePair(ind = index_sets[i], val = [1] * len(index_sets[i])) for i in range(n)], 
            senses = ["L"] * n,
            rhs = N_cells) #Defining the "don't share too much" constraint

            m.linear_constraints.add(lin_expr = [cplex.SparsePair(ind = range(N), val = [-(len(X[i]) - 1) for i in range(N)])],
                senses = ["L"],
                rhs = [F - sum(N_cells)])



            m.parameters.optimalitytarget.set(3)
            m.set_results_stream(None)
            m.set_log_stream(None)

            #Optimising
            m.solve()   

            if m.solution.get_status() !=  119: #This is the status code for infeasible solution. Occasionally the solver fails due to its pre-processing methods.
                optim_Os = np.array(m.solution.get_values())
                optim_T = m.solution.get_objective_value()
            else: 
                optim_Os = np.zeros(N) #Assigning a global minima instead. Other solvers will then overrule this one. 
                optim_T = -1 #Assigning a value which will NOT be selected instead of the other solvers.
                
            #Storing data
            optims[j][i] = optim_Os  
            T_vals[j, i] = optim_T
            individual_costs[j][i] = get_shared_and_unique(optim_Os, n, N_cells, X)[0]
                
            for k in range(2, n+1):
                w_vals[j, i, k-2] += get_w_i_n(X, optim_Os, N_cells, N, k, n)

    costs_different = [[None] * num_Ds for i in range(num_Fs)] 
    costs_normal = [[None] * num_Ds for i in range(num_Fs)]
    
    for i in range(num_Ds):
        for j in range(num_Fs):
            costs_different[j][i] = individual_costs[j][i][0]
            costs_normal[j][i] = individual_costs[j][i][1]

    return w_vals, optims, costs_different, costs_normal, T_vals

#Best of the three above
def ONE_DIFF_INTEGER_OPTIMISE_FORAGE_best(n, N_cell_normal, D_min, D_max, num_Ds, F_min, F_max, num_Fs):
    #Optimise with each solver and return the best solution for each tested value of D and F
    costs_different = [[None] * num_Ds for i in range(num_Fs)]
    costs_normal = [[None] * num_Ds for i in range(num_Fs)]
    optims = [[None] * num_Ds for i in range(num_Fs)]
    ws = np.zeros([num_Fs, num_Ds, n-1])
    Ts = np.zeros((num_Fs, num_Ds))

    T_vals_all = [None] * 3
    ws_all = [None] * 3
    optims_all = [None] * 3
    costs_different_all = [None] * 3
    costs_normal_all = [None] * 3

    print("With Gurobi...")
    ws_all[0], optims_all[0], costs_different_all[0], costs_normal_all[0], T_vals_all[0] = ONE_DIFF_INTEGER_OPTIMISE_FORAGE_gp(n, N_cell_normal, D_min, D_max, num_Ds, F_min, F_max, num_Fs) 
    print("With Xpress...")
    ws_all[1], optims_all[1], costs_different_all[1], costs_normal_all[1], T_vals_all[1] = ONE_DIFF_INTEGER_OPTIMISE_FORAGE_xp(n, N_cell_normal, D_min, D_max, num_Ds, F_min, F_max, num_Fs) 
    print("With CPLEX...")
    ws_all[2], optims_all[2], costs_different_all[2], costs_normal_all[2], T_vals_all[2] = ONE_DIFF_INTEGER_OPTIMISE_FORAGE_cplex(n, N_cell_normal, D_min, D_max, num_Ds, F_min, F_max, num_Fs) 

    print("Comparing results...")
    for i in range(num_Ds):
        for j in range(num_Fs):
            T_vals = [T_vals_all[0][j,i], T_vals_all[1][j,i], T_vals_all[2][j,i]]
            Ts[j, i] = max(T_vals)
            best_index = T_vals.index(Ts[j, i])

            costs_different[j][i] = costs_different_all[best_index][j][i]
            costs_normal[j][i] = costs_normal_all[best_index][j][i]
            ws[j, i, :] = ws_all[best_index][j, i, :]
            optims[j][i] = optims_all[best_index][j][i]

    
    return ws, optims, costs_different, costs_normal, Ts

##### Plotting

In [ ]:

def plot_T_with_F_D(T_vals, D_min, D_max, num_Ds, F_min, F_max, num_Fs, N_cell_normal, show_lines = False):
    #Plotting function for T values over F and D
    Fs = np.linspace(F_min, F_max, num_Fs)
    Ds = np.linspace(D_min, D_max, num_Ds)

    fig = make_subplots(
        rows=1, cols=1)
    
    fig.add_trace(go.Contour(
        z=np.round(T_vals,10),
        x=Ds, 
        y=Fs,
        contours=dict(
            start=0,
            end=np.max(T_vals),
            size=np.max(T_vals) / 20,
        ), colorscale='viridis'
    ))
    
    fig.update_layout(
    autosize=False,
    width=480,
    height=350,
    margin=dict(
        l=10,
        r=10,
        b=10,
        t=10,
        pad=2
    ))
    
    if show_lines:
        line_grad = N_cell_normal 
        line_intercept = 0
        fig.add_scatter(
            x=[D_min, D_max], 
            y=[N_cell_normal, N_cell_normal], 
            mode='lines', 
            line_color='white', 
            line_dash = 'dash',
            showlegend=False
        )
        
        fig.add_scatter(
            x=[(F_min - line_intercept) / line_grad, (F_max-line_intercept)/line_grad], 
            y=[F_min, F_max], 
            mode='lines', 
            line_color='white', 
            line_dash = 'dash',
            showlegend=False
        )

    fig.update_yaxes(title_text=f"Available resource level, F")
    fig.update_xaxes(title_text=f"Scale difference, D")
    
    return fig

def plot_ws_with_F_D(n, w_vals, D_min, D_max, num_Ds, F_min, F_max, num_Fs, N_cell_normal, show_lines = False):
    #Plotting function for w_n^(i) values over F and D
    Fs = np.linspace(F_min, F_max, num_Fs)
    Ds = np.linspace(D_min, D_max, num_Ds)    
    fig = make_subplots(rows=1, cols=n-1)

    fig.update_layout(
        autosize=False,
        width=300*n,
        height=350,
        margin=dict(
            l=10,
            r=10,
            b=10,
            t=10,
            pad=2
        ))
    
    for size in range(2, n+1):
        fig.add_trace(go.Contour(
                z=np.round(w_vals[:, :, size-2], 10),
                x=Ds,
                y=Fs, 
                contours=dict(
                    start=0,
                    end=1,
                    size=1 / 20,
                ), line_smoothing=0
            ),row=1, col=size-1)

    
    if show_lines:
        line_grad = N_cell_normal
        line_intercept = 0
        for s in range(1, n):
            fig.add_scatter(
                x=[D_min, D_max], 
                y=[N_cell_normal, N_cell_normal], 
                mode='lines', 
                line_color='white', 
                line_dash = 'dash',
                showlegend=False, row=1, col=s
            )
            
            fig.add_scatter(
                x=[(F_min - line_intercept ) / line_grad, (F_max-line_intercept)/line_grad], 
                y=[F_min, F_max], 
                mode='lines', 
                line_color='white', 
                line_dash = 'dash',
                showlegend=False, row=1, col=s
            )

    fig.update_yaxes(title_text=f"Available resource level, F", row=1, col=1)

    for i in range(1, n):
        fig.update_xaxes(title_text=f"Scale difference, D", row=1, col=i)
    

    return fig

def plot_costs_w_F_D(costs_different, costs_normal, D_min, D_max, num_Ds, F_min, F_max, num_Fs, N_cell_normal, show_lines = False):
    #Plotting function for C_1 and C_k (k>1) over F and D
    Fs = np.linspace(F_min, F_max, num_Fs)
    Ds = np.linspace(D_min, D_max, num_Ds)

    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{}, {}]], subplot_titles=("Different individual sharing","Regular individual sharing"))
    
    fig.update_layout(
        autosize=False,
        width=900,
        height=350,
        margin=dict(
            l=10,
            r=10,
            b=10,
            t=25,
            pad=2
        ))

    fig.add_trace(go.Contour(
            z=np.round(costs_different, 8),
            x=Ds, 
            y=Fs,
            contours=dict(
                start=0,
                end=1,
                size=1 / 20 ,
            ), colorscale='rdylbu_r'#'earth_r'
        ),row=1, col=1)

    fig.add_trace(go.Contour(
            z=np.round(costs_normal, 8),
            x=Ds, 
            y=Fs,
            contours=dict(
                start=0,
                end=1,
                size=1/20,
            ), colorscale='rdylbu_r'#'earth_r'
        ),row=1, col=2)


    if show_lines:
        line_grad = N_cell_normal
        line_intercept = 0
        for s in [1, 2]:
            fig.add_scatter(
                x=[D_min, D_max], 
                y=[N_cell_normal, N_cell_normal], 
                mode='lines', 
                line_color='white', 
                line_dash = 'dash',
                showlegend=False, row=1, col=s
            )
            
            fig.add_scatter(
                x=[(F_min - line_intercept ) / line_grad, (F_max-line_intercept)/line_grad], 
                y=[F_min, F_max], 
                mode='lines', 
                line_color='white', 
                line_dash = 'dash',
                showlegend=False, row=1, col=s
            )

    fig.update_yaxes(title_text=f"Available resource level, F", row=1, col=1)
    fig.update_xaxes(title_text=f"Scale difference, D", row=1, col=1)
    fig.update_xaxes(title_text=f"Scale difference, D", row=1, col=2)

    return fig

def plot_w_weighted_means_FD(n, ws, D_min, D_max, num_Ds, F_min, F_max, num_Fs, show_lines = False):
    #Plotting function for K over F and D
    w_means = np.zeros((num_Fs, num_Ds))
    for i in range(num_Ds):
        for j in range(num_Fs):
            w_means[j, i] = get_w_weight_mean(n, ws[j, i, :])


    Fs = np.linspace(F_min, F_max, num_Fs)
    Ds = np.linspace(D_min, D_max, num_Ds)

    fig = make_subplots(
        rows=1, cols=1)

    fig.add_trace(go.Contour(
        z=w_means,
        x=Ds, 
        y=Fs,
        contours=dict(
            start=1,
            end=n,
            size= (n-1) / 20,
        ), colorscale='oranges'
    ))

    if show_lines:
        line_grad = N_cell_normal 
        line_intercept = 0
        fig.add_scatter(
            x=[D_min, D_max], 
            y=[N_cell_normal, N_cell_normal], 
            mode='lines', 
            line_color='white', 
            line_dash = 'dash',
            showlegend=False
        )
        
        fig.add_scatter(
            x=[(F_min - line_intercept) / line_grad, (F_max-line_intercept)/line_grad], 
            y=[F_min, F_max], 
            mode='lines', 
            line_color='white', 
            line_dash = 'dash',
            showlegend=False
        )

    fig.update_layout(
    autosize=False,
    width=480,
    height=350,
    margin=dict(
        l=10,
        r=10,
        b=10,
        t=10,
        pad=2
    ))

    return fig

##### Running model

In [ ]:
n = 5 #Number of individuals
N_cell_normal = 10000 #Number of cells each individual can occupy
D_min = 0.01 #Scaling for worst forager considered
D_max = 4 #Scaling for best 'different forager' considered
num_Ds = 50 #Number of different ability scalings considered
F_min = 100#Lowest number of maximum foraging points considered 
F_max =  40000#Highest number of maximum foraging points considered 
num_Fs = 50 #int((F_max-F_min+1)/200+1) #Number of different foraging constraints considered

Fs = np.linspace(F_min, F_max, num_Fs)
Ds = np.linspace(D_min, D_max, num_Ds)
want_lines = False
    
ws, optims, costs_different, costs_normal, Ts = ONE_DIFF_INTEGER_OPTIMISE_FORAGE_best(n, N_cell_normal, D_min, D_max, num_Ds, F_min, F_max, num_Fs)

#The maxima shows that the strong diadic optimality at certain F values is specific to homogeneous systems
#Seems like there is _always_ a peak in the lowest order interactions


##### Plotting results

In [ ]:
want_lines = False

fig = plot_T_with_F_D(Ts, D_min, D_max, num_Ds, F_min, F_max, num_Fs, N_cell_normal, want_lines)
fig.show()

fig = plot_ws_with_F_D(n, np.round(ws, 3), D_min, D_max, num_Ds, F_min, F_max, num_Fs, N_cell_normal, want_lines)
fig.update_layout(
        autosize=False,
        width=1100,
        height=350,
        margin=dict(
            l=10,
            r=10,
            b=10,
            t=25,
            
            pad=2
        ))
fig.show()

fig_ = plot_costs_w_F_D(np.round(costs_different, 3), np.round(costs_normal, 3), D_min, D_max, num_Ds, F_min, F_max, num_Fs, N_cell_normal, want_lines)
fig_.update_layout(
        autosize=False,
        height=350,
        width = 1100,
        margin=dict(
            l=10,
            r=10,
            b=10,
            t=25,
            
            pad=2
        ))
fig_.show()

fig = plot_w_weighted_means_FD(n, np.round(ws,4), D_min, D_max, num_Ds, F_min, F_max, num_Fs, want_lines)
fig.show()


In [ ]:
fig = make_subplots(
        rows=1, cols=2,
        specs=[[{}, {}]], subplot_titles = ["Group information transfer, T", "Average point knowledge, K"])

w_means = np.zeros((num_Fs, num_Ds))
for i in range(num_Ds):
        for j in range(num_Fs):
                w_means[j, i] = get_w_weight_mean(n, ws[j, i, :])


Fs = np.linspace(F_min, F_max, num_Fs)
Ds = np.linspace(D_min, D_max, num_Ds)

fig.add_trace(go.Contour(
        z=np.round(Ts,10),
        x=Ds, 
        y=Fs,
        contours=dict(
            start=0,
            end=np.max(Ts),
            size=np.max(Ts) / 20,
        ), colorscale='viridis'
    ), row=1, col=1)
    
fig.add_trace(go.Contour(
z=w_means,
x=Ds, 
y=Fs,
contours=dict(
        start=1,
        end=n,
        size= (n-1) / 20,
), colorscale='thermal_r'
),row=1, col=2)


fig.update_layout(
        autosize=False,
        width=1100,
        height=350,
        margin=dict(
            l=10,
            r=10,
            b=10,
            t=25,
            
            pad=2
        ))

for s in [1,2]:
        line_grad = N_cell_normal 
        line_intercept = 0
        fig.add_scatter(
                x=[D_min, D_max], 
                y=[N_cell_normal, N_cell_normal], 
                mode='lines', 
                line_color='white', 
                line_dash = 'dash',
                showlegend=False, row=1, col=s
        )

        fig.add_scatter(
                x=[(F_min - line_intercept) / line_grad, (F_max-line_intercept)/line_grad], 
                y=[F_min, F_max], 
                mode='lines', 
                line_color='white', 
                line_dash = 'dash',
                showlegend=False, row=1, col=s
        )

fig.show()

fig.update_yaxes(title_text=f"Available resource level, F", row=1, col=1)
fig.update_yaxes(title_text=f"Available resource level, F", row=1, col=2)

fig.update_xaxes(title_text=f"Scale difference, D", row=1, col=1)
fig.update_xaxes(title_text=f"Scale difference, D", row=1, col=2)



## Scenario 3: Heterogeneous 


#### Knowledge distribution functions

In [ ]:

def get_pretruncated_stats(N_mean, sigma):
    #Code for getting the corrrect intitial distribution values for the post-truncation interval to have mean N_mean and SD sigma, as described in the supplementary information.
    def truncated_stats(x):
        def alpha(x):
            return (0 - x[0]) / x[1]

        def psi(alp):
            return 1/(np.sqrt(2*np.pi)) * np.exp(-0.5*alp**2) 
        
        def phi(alp):
            return 0.5 * (1 + scipy.special.erf(alp/np.sqrt(2)))

        #x[0] is the mean of the distribution before truncation, x[1] is the SD and x[2] is the lower truncation point. The upper truncation point is +infinity.
        #First output is the mean after truncation minus the deisred and the second is the SD after truncation minus the desired
        alp = alpha(x)
        return [x[0]+(psi(alp) / (1-phi(alp)))*x[1] - N_mean, x[1]*np.sqrt(1 + (alp*psi(alp) / (1-phi(alp))) - (psi(alp) / (1-phi(alp)))**2) - sigma]

    root = scipy.optimize.root(truncated_stats, [N_mean, sigma])
    
    return root.x

def get_N_cells_norm(n, N_mean, num_sims_per_sigma, sigma_min, sigma_max, num_sigmas, fixed_points = False): #Minimally skewed distribution, with lower bound at 0.5 to avoid potential division by zero errors. 
    #Generate truncated normal distributed N_cells for n individuals, over a range of standard deviations.
    N_cells_all = np.zeros((num_sigmas, num_sims_per_sigma, n))
    sigmas = np.linspace(sigma_min, sigma_max, num_sigmas) 
    for i in range(num_sigmas):
        sigma = sigmas[i]
        if sigma != 0:
            initial_stats = get_pretruncated_stats(N_mean-0.5, sigma)
            a, b = (0 - initial_stats[0]) / initial_stats[1], (np.inf - initial_stats[0]) / initial_stats[1]
            for j in range(num_sims_per_sigma):
                N_cells = scipy.stats.truncnorm.rvs(a, b, loc=initial_stats[0], scale=initial_stats[1], size=n) + 0.5 #Translating to have correct mean and variance but have minimum of 0.5
                if fixed_points: #If we want to normalise, as presented in the supporting information.
                    N_cells = n*N_mean * N_cells / sum(N_cells)
                N_cells_all[i, j, :] = np.round(N_cells) 
        else: #sigma is zero - homogeneous case
            N_cells_all[i, :, :] = N_mean*np.ones((num_sims_per_sigma, n)) 
    return N_cells_all

def get_N_cells_log_norm(n, N_mean, num_sims_per_sigma, sigma_min, sigma_max, num_sigmas, fixed_points = False): #Minimally skewed distribution, with lower bound at 0.5 to avoid potential division by zero errors. 
    #Generate truncated normal distributed N_cells for n individuals, over a range of standard deviations.
    N_cells_all = np.zeros((num_sigmas, num_sims_per_sigma, n))
    sigmas = np.linspace(sigma_min, sigma_max, num_sigmas) 
    for i in range(num_sigmas):
        sigma = sigmas[i]
        if sigma != 0:
            s_ = np.sqrt(np.log(1 + (sigma**2 / (N_mean-0.5)**2)))
            mu = np.log(N_mean-0.5) - (s_**2 / 2)
            for j in range(num_sims_per_sigma):
                N_cells = scipy.stats.lognorm.rvs(s=s_, scale=np.exp(mu), size=n) + 0.5
                if fixed_points: #If we want to normalise, as presented in the supporting information.
                    N_cells = n*N_mean * N_cells / sum(N_cells)
                N_cells_all[i, j, :] = np.round(N_cells) 
        else: #sigma is zero - homogeneous case
            N_cells_all[i, :, :] = N_mean*np.ones((num_sims_per_sigma, n)) 
            
    return N_cells_all



#### Without foraging constraint

##### Optimisation code

In [ ]:
def GENERAL_VAR_INTEGER_OPTIMISE_gp(n, N_cells_all, num_sims_per_sigma, num_sigmas):
    # Optimise in the case with n individuals with general foraging abilities, set up to be in order of increasing variance. Using Gurobi solver.   
    with gp.Env(empty=True) as env: #To avoid text showing optimisation procedure. 
        env.setParam('OutputFlag', 0)
        env.start()

        #Pre-allocate space and define ranges
        individual_costs = [[None] * num_sims_per_sigma for i in range(num_sigmas)]
        optims =  [[None] * num_sims_per_sigma for i in range(num_sigmas)]
        w_vals = np.zeros([num_sigmas, num_sims_per_sigma, n-1])
        T_vals = np.zeros([num_sigmas, num_sims_per_sigma])

        
        #Setup independent to m
        X, N, X_sizes_increasing, X_sizes_decreasing = find_subsets(n)
        
        N_diff = (2**(n-1)) - 1
        entries = np.ones(n*N_diff)
        
        row_indices = np.zeros(n * N_diff)
        col_indices = [None] * (n * N_diff)
        for i in range(n):
            col_indices[i*N_diff:(i+1)*N_diff] = [X.index(set_) for set_ in X if {i}.issubset(set_)]
            row_indices[i*N_diff:(i+1)*N_diff] = i 
            
        col_indices = np.array(col_indices)
    
        G = sp.csr_matrix((entries, (row_indices, col_indices)), shape=(n, N))
    
    
        #setup involving m, and optimisation
        for j in range(num_sims_per_sigma):
            print(j)
            for i in range(num_sigmas):
                #Initialise the model
                m = gp.Model(env = env, name = "test")
                
                #Define the variables (the overlapping points)
                O = m.addMVar(shape=N, vtype=GRB.INTEGER, name="O") #Constraining the solutions to be integers
        
                #Adding constraints
                m.addConstrs((O[i] >= 0 for i in range(N)), name = 'non-neg bounds') #For non-negativity
                
                N_cells = N_cells_all[i, j, :]
                
                #Might want to track the N values themsevles? So we can plot the empirical variances.
                    
                h = np.array(N_cells)
                
                m.addConstr(G @ O <= h, name='sharing') #Defining the "don't share too much" constraint
        
                #Defining the objective function
                S_sets= [set(S(c, X, X_sizes_decreasing, N)) for c in range(N)]
                B_sets = [set(B(c, X, X_sizes_increasing, N)) for c in range(N)]
                sum_Ns, prod_Ns = give_sums_and_prods(X, N, N_cells)
                l = get_l(N, S_sets, sum_Ns, prod_Ns)
                M = get_M(X, N, S_sets, B_sets, prod_Ns)
                
                m.setObjective(l@O + 0.5 * O.T @ M @ O, GRB.MAXIMIZE)
    
                #Optimising 
                m.setParam('TimeLimit', 180) #3 mins max getting to a solution
                m.setParam('MIPGap', 0.0001) #Setting a looser bound on optimality
                m.optimize()
    
                #Storing data
                if m.SolCount > 0:
                    optim_Os = O.X
                    optims[i][j] = optim_Os
                    T_vals[i, j] = m.ObjVal
                    individual_costs[i][j] = get_shared_and_unique(optim_Os, n, N_cells, X)[0]
                        
                    for k in range(2, n+1):
                        w_vals[i, j, k-2] += get_w_i_n(X, optim_Os, N_cells, N, k, n)
                else: 
                    print(f"No solution found for sigma index {i} and sim index {j}")
                    T_vals[i, j] = -np.inf #Indicating no solution found with negative infinity, so we can easily go with the other solvers

    return w_vals, optims, individual_costs, T_vals

def GENERAL_VAR_INTEGER_OPTIMISE_xp(n, N_cells_all, num_sims_per_sigma, num_sigmas):
    # Optimise in the case with n individuals with general foraging abilities, set up to be in order of increasing variance. Using Xpress solver. 
    xp.controls.outputlog = 0
    xp.setOutputEnabled(False)
    
    #Pre-allocate space and define ranges
    individual_costs = [[None] * num_sims_per_sigma for i in range(num_sigmas)]
    optims =  [[None] * num_sims_per_sigma for i in range(num_sigmas)]
    w_vals = np.zeros([num_sigmas, num_sims_per_sigma, n-1])
    T_vals = np.zeros([num_sigmas, num_sims_per_sigma])

    
    #Setup independent to m
    X, N, X_sizes_increasing, X_sizes_decreasing = find_subsets(n)
    index_sets = [[X.index(set_) for set_ in X if {i}.issubset(set_)] for i in range(n)]


    #setup involving m, and optimisation
    for j in range(num_sims_per_sigma):
        print(j)
        for i in range(num_sigmas):
            #Initialise the model
            m = xp.problem()
            
            #Define the variables (the overlapping points)
            O = np.array([m.addVariable(lb=0, vartype = xp.integer, name = f'O_{i+1}') for i in range(N)]) #Constraining the solutions to be integers
    
            #Defining the objective
            N_cells = N_cells_all[i, j, :]

            S_sets= [set(S(c, X, X_sizes_decreasing, N)) for c in range(N)]
            B_sets = [set(B(c, X, X_sizes_increasing, N)) for c in range(N)]
            sum_Ns, prod_Ns = give_sums_and_prods(X, N, N_cells)

            l = get_l(N, S_sets, sum_Ns, prod_Ns)
            M = get_M(X, N, S_sets, B_sets, prod_Ns)
                
            m.setObjective(l@O + 0.5 * O.T @ M @ O, sense = xp.maximize)

            #Adding constraints
            for a in range(n):
                m.addConstraint(xp.Sum(O[index_sets[a]]) <= N_cells[a])
            
            #Optimising 
            m.controls.timelimit = 180 #3 mins max getting to a solution
            m.optimize()

            #Storing data
            optim_Os = np.array(m.getSolution())
            optims[i][j] = optim_Os
            T_vals[i, j] = l@optim_Os + 0.5 * optim_Os.T @ M @ optim_Os
            individual_costs[i][j] = get_shared_and_unique(optim_Os, n, N_cells, X)[0]
                
            for k in range(2, n+1):
                w_vals[i, j, k-2] += get_w_i_n(X, optim_Os, N_cells, N, k, n)


    return w_vals, optims, individual_costs, T_vals

def GENERAL_VAR_INTEGER_OPTIMISE_cplex(n, N_cells_all, num_sims_per_sigma, num_sigmas):
    # Optimise in the case with n individuals with general foraging abilities, set up to be in order of increasing variance. Using CPLEX solver.   
    
    #Pre-allocate space and define ranges
    individual_costs = [[None] * num_sims_per_sigma for i in range(num_sigmas)]
    optims =  [[None] * num_sims_per_sigma for i in range(num_sigmas)]
    w_vals = np.zeros([num_sigmas, num_sims_per_sigma, n-1])
    T_vals = np.zeros([num_sigmas, num_sims_per_sigma])

    
    #Setup independent to m
    X, N, X_sizes_increasing, X_sizes_decreasing = find_subsets(n)
    S_sets= [set(S(c, X, X_sizes_decreasing, N)) for c in range(N)]
    B_sets = [set(B(c, X, X_sizes_increasing, N)) for c in range(N)]
    index_sets = [[X.index(set_) for set_ in X if {i}.issubset(set_)] for i in range(n)]

    #setup involving m, and optimisation
    for j in range(num_sims_per_sigma):
        print(j)
        for i in range(num_sigmas):
            #Initialise the model
            m = cplex.Cplex() 
            m.set_problem_type(m.problem_type.MIQP)
            m.objective.set_sense(m.objective.sense.maximize)
            
            #Defining the objective and the variables
            
    
            N_cells = N_cells_all[i, j, :]

            sum_Ns, prod_Ns = give_sums_and_prods(X, N, N_cells)

            l = get_l(N, S_sets, sum_Ns, prod_Ns)
            M = get_M(X, N, S_sets, B_sets, prod_Ns)
                
            O = m.variables.add(obj=l, types=[m.variables.type.integer] * N, lb = [0] * N)
            m.objective.set_quadratic([cplex.SparsePair(ind = range(N), val = [M[i, k] for k in range(N)]) for i in range(N)])

            #Adding constraints
            m.linear_constraints.add(
                lin_expr = [cplex.SparsePair(ind = index_sets[i], val = [1] * len(index_sets[i])) for i in range(n)], 
                senses = ["L"] * n,
                rhs = N_cells) #Defining the "don't share too much" constraint

            m.parameters.optimalitytarget.set(3)
            m.set_results_stream(None)
            m.set_log_stream(None)
            m.set_warning_stream(None)
            m.parameters.timelimit.set(180) #3 min max 

            
            #Optimising 
            m.solve()

            if m.solution.get_status() !=  119: #This is the status code for infeasible solution. Occasionally the solver fails due to its pre-processing methods.
                optim_Os = np.array(m.solution.get_values())
                optim_T = m.solution.get_objective_value()
            else: 
                optim_Os = np.zeros(N) #Assigning a global minima instead. Other solvers will then overrule this one. 
                optim_T = -1 #Assigning a value which will NOT be selected instead of the other solvers
                    
            #Storing data
            optims[i][j] = optim_Os
            T_vals[i, j] = optim_T
            individual_costs[i][j] = get_shared_and_unique(optim_Os, n, N_cells, X)[0]
                
            for k in range(2, n+1):
                w_vals[i, j, k-2] += get_w_i_n(X, optim_Os, N_cells, N, k, n)


    return w_vals, optims, individual_costs, T_vals

def GENERAL_VAR_INTEGER_OPTIMISE_best(n, N_cells_all, num_sims_per_sigma, num_sigmas):
    #Optimise with each solver and return the best solution for each tested value of sigma
    individual_costs = [[None] * num_sims_per_sigma for i in range(num_sigmas)]
    optims =  [[None] * num_sims_per_sigma for i in range(num_sigmas)]
    ws = np.zeros([num_sigmas, num_sims_per_sigma, n-1])
    Ts = np.zeros([num_sigmas, num_sims_per_sigma])

    T_vals_all = [None] * 3
    ws_all = [None] * 3
    optims_all = [None] * 3
    individual_costs_all = [None] * 3

    print("With Gurobi...")
    ws_all[0], optims_all[0], individual_costs_all[0], T_vals_all[0] = GENERAL_VAR_INTEGER_OPTIMISE_gp(n, N_cells_all, num_sims_per_sigma, num_sigmas) 
    print("With Xpress...")
    ws_all[1], optims_all[1], individual_costs_all[1], T_vals_all[1] = GENERAL_VAR_INTEGER_OPTIMISE_xp(n, N_cells_all, num_sims_per_sigma, num_sigmas)
    print("With CPLEX...")
    ws_all[2], optims_all[2], individual_costs_all[2], T_vals_all[2] = GENERAL_VAR_INTEGER_OPTIMISE_cplex(n, N_cells_all, num_sims_per_sigma, num_sigmas)

    print("Comparing results...")
    for i in range(num_sigmas):
        for j in range(num_sims_per_sigma):
            T_vals = [T_vals_all[0][i,j], T_vals_all[1][i,j], T_vals_all[2][i,j]]
            Ts[i, j] = max(T_vals)
            best_index = T_vals.index(Ts[i, j])

            individual_costs[i][j] = individual_costs_all[best_index][i][j]
            ws[i, j, :] = ws_all[best_index][i, j, :]
            optims[i][j] = optims_all[best_index][i][j]

    return ws, optims, individual_costs, Ts

##### Functions for plotting

In [ ]:
def plot_ws_with_vars(n, w_vals, T_vals, sigma_min, sigma_max, num_sigmas, num_sims_per_sigma):

    ws_mean = np.mean(w_vals, axis=1)
    T_vals_mean = np.mean(T_vals, axis=1)

    fig, ax = plt.subplots(1, n)

    sigmas = np.linspace(sigma_min, sigma_max, num_sigmas)

    for q in range(n-1):
        ws_mean_ = ws_mean[:, q]
        slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(sigmas, ws_mean_)
        R_squared = r_value ** 2
        poly1d_fn = np.poly1d([slope, intercept]) 
        for j in range(num_sims_per_sigma):
            ax[q].plot(sigmas, w_vals[:, j, q], 'x', color = (0.3, 0.2, 0.8, 0.2))

        ax[q].plot(sigmas, poly1d_fn(sigmas), '--k', label = f"LoBF $R^2 = ${round(R_squared,4)}")
        ax[q].plot(sigmas, ws_mean_, 'yo', label = "MC Points")
        ax[q].set_ylim([0-0.01, np.max(w_vals[:, :, :])+0.02])
        ax[q].legend(loc = 'upper left')
        ax[q].set_xlabel('Variance $\\sigma$')
        ax[q].set_ylabel(f'$w_{n}^{q+2}$')
        # ax[q].set_ylim((0, 1))

    ax[-2].legend(loc = 'lower left')

    for j in range(num_sims_per_sigma):
        ax[-1].plot(sigmas, T_vals[:, j], 'x', color = (0.5, 0.8, 0.2, 0.2))
    ax[-1].plot(sigmas, T_vals_mean, 'o', label = "MC Points")
    ax[-1].set_xlabel('Variance $\\sigma$')
    ax[-1].set_ylabel('$T(O^*)$')
    ax[-1].legend(loc = 'upper left')

    fig.set_figwidth(7*n)
    fig.set_figheight(5)

    return fig, ax

def plot_sigmas_with_vars(n, ws, Ts, sigma_min, sigma_max):

    ws_vars = np.var(ws, axis=1)
    T_vals_vars = np.var(Ts, axis=1)

    fig, ax = plt.subplots(1, n)

    sigmas = np.linspace(sigma_min, sigma_max, num_sigmas)

    for q in range(n-1):
        ws_vars_ = ws_vars[:, q]
        slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(sigmas, ws_vars_)
        R_squared = r_value ** 2
        poly1d_fn = np.poly1d([slope, intercept]) 
        
        ax[q].plot(sigmas, poly1d_fn(sigmas), '--k', label = f"LoBF $R^2 = ${round(R_squared,4)}")
        ax[q].plot(sigmas, ws_vars_, 'yo', label = "MC Points")
        #ax[q].set_ylim([0-0.01, np.max(w_vals[:, :, :])+0.02])
        ax[q].legend(loc = 'upper left')
        ax[q].set_xlabel('Variance $\\sigma$')
        ax[q].set_ylabel(f'$V(w_{n}^{q+2})$')
        # ax[q].set_ylim((0, 1))

    # ax[-2].legend(loc = 'lower left')

    slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(sigmas, T_vals_vars)
    R_squared = r_value ** 2
    poly1d_fn = np.poly1d([slope, intercept]) 

    ax[-1].plot(sigmas, T_vals_vars, 'o', label = "MC Points")
    ax[-1].plot(sigmas, poly1d_fn(sigmas), '--k', label = f"LoBF $R^2 = ${round(R_squared,4)}")

    ax[-1].set_xlabel('Variance $\\sigma$')
    ax[-1].set_ylabel('$V(T(O^*))$')
    ax[-1].legend(loc = 'upper left')

    fig.set_figwidth(7*n)
    fig.set_figheight(5)

    return fig, ax

def plot_mean_var_general(n, ws, Ts):    
    fig, ax = plt.subplots(1, n)
    
    ws_mean = np.mean(ws, axis=1)
    ws_vars = np.var(ws, axis=1)
    T_vals_mean = np.mean(Ts, axis=1)
    T_vals_vars = np.var(Ts, axis=1)

    for q in range(n-1):
        
        ws_vars_ = ws_vars[:, q]
        ws_mean_ = ws_mean[:, q]


        slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(ws_mean_, ws_vars_)
        R_squared = r_value ** 2
        poly1d_fn = np.poly1d([slope, intercept]) 
        
        ax[q].plot(ws_mean_, poly1d_fn(ws_mean_), '--k', label = f"LoBF $R^2 = ${round(R_squared,4)}")
        ax[q].plot(ws_mean_, ws_vars_, 'yo', label = "MC Points")
        #ax[q].set_ylim([0-0.01, np.max(w_vals[:, :, :])+0.02])
        ax[q].legend(loc = 'upper left')
        ax[q].set_xlabel(f'Mean of $w_{n}^{q+2}$')
        ax[q].set_ylabel(f'Variance of $w_{n}^{q+2})$')
        # ax[q].set_ylim((0, 1))

    # ax[-2].legend(loc = 'lower left')

    slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(T_vals_mean, T_vals_vars)
    R_squared = r_value ** 2
    poly1d_fn = np.poly1d([slope, intercept]) 

    ax[-1].plot(T_vals_mean, T_vals_vars, 'o', label = "MC Points")
    ax[-1].plot(T_vals_mean, poly1d_fn(T_vals_mean), '--k', label = f"LoBF $R^2 = ${round(R_squared,4)}")

    ax[-1].set_xlabel(f'Mean of $T(O^*)$')
    ax[-1].set_ylabel('Variance of $T(O^*)$')
    ax[-1].legend(loc = 'upper left')

    fig.set_figwidth(7*n+5)
    fig.set_figheight(5)

    return fig, ax    

def plot_w_weighted_mean(n, ws, num_sims_per_sigma, sigma_min, sigma_max, num_sigmas):
        
    sigmas = np.linspace(sigma_min, sigma_max, num_sigmas)
    Ks = np.zeros((num_sims_per_sigma, num_sigmas))
    for j in range(num_sims_per_sigma):
        for i in range(num_sigmas):
            Ks[j, i] = get_w_weight_mean(n, ws[i, j, :])
    
    Ks_mean = np.mean(Ks, axis = 0)
    slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(sigmas, Ks_mean)
    R_squared = r_value ** 2
    poly1d_fn = np.poly1d([slope, intercept])

    
    for j in range(num_sims_per_sigma):
        plt.plot(sigmas, Ks[j, :], 'x', color = (0/255, 108/255, 209/255, 0.2))

    plt.plot(sigmas, Ks_mean, 'o', color = '#994F00')    
    plt.plot(sigmas, poly1d_fn(sigmas), '--k', label = f"LoBF $R^2 = ${round(R_squared,4)}")

    
    plt.xlabel("Variance $\\sigma$")
    plt.ylabel("Average point knowledge, $K$")
    return fig


##### Some results
To avoid kernal crashes on my machine, we run the model in batches of 50 sigma values. Saved as we go along into different files. 

In [ ]:
n = 5 #Number of individuals
N_mean = 10000 #Number of cells each individual can occupy
sigma_min = 0 #Lowest SD in foraging abilities considered
sigma_max = N_mean*3 #Highest SD in foraging abilities considered - extreme overdispersion - longer verision
num_sigmas = 250 #Number of different variances
num_sims_per_sigma = 100 #Number of MC simulations per combination of parameters

N_cells_all_norm = get_N_cells_norm(n, N_mean, num_sims_per_sigma, sigma_min, sigma_max, num_sigmas, False)
N_cells_all_log_norm = get_N_cells_log_norm(n, N_mean, num_sims_per_sigma, sigma_min, sigma_max, num_sigmas, False)


print("Truncated normal distribution...")
print("Batch 1")
ws__1_norm_no_F, optims__1_norm_no_F, individual_costs__1_norm_no_F, Ts__1_norm_no_F = GENERAL_VAR_INTEGER_OPTIMISE_best(n, N_cells_all_norm[:, 0:int(num_sims_per_sigma/5), :], int(num_sims_per_sigma/5), num_sigmas)
np.save("ws1_norm_no_F_longer.npy", ws__1_norm_no_F)
np.save("optims1_norm_no_F_longer.npy", optims__1_norm_no_F)
np.save("indv_costs1_norm_no_F_longer.npy", individual_costs__1_norm_no_F)
np.save("Ts1_norm_no_F_longer.npy", Ts__1_norm_no_F)

print("Batch 2")
ws__2_norm_no_F, optims__2_norm_no_F, individual_costs__2_norm_no_F, Ts__2_norm_no_F = GENERAL_VAR_INTEGER_OPTIMISE_best(n, N_cells_all_norm[:, int(num_sims_per_sigma/5):2*int(num_sims_per_sigma/5), :], int(num_sims_per_sigma/5), num_sigmas)
np.save("ws2_norm_no_F_longer.npy", ws__2_norm_no_F)
np.save("optims2_norm_no_F_longer.npy", optims__2_norm_no_F)
np.save("indv_costs2_norm_no_F_longer.npy", individual_costs__2_norm_no_F)
np.save("Ts2_norm_no_F_longer.npy", Ts__2_norm_no_F)

print("Batch 3")
ws__3_norm_no_F, optims__3_norm_no_F, individual_costs__3_norm_no_F, Ts__3_norm_no_F = GENERAL_VAR_INTEGER_OPTIMISE_best(n, N_cells_all_norm[:, 2*int(num_sims_per_sigma/5):3*int(num_sims_per_sigma/5), :], int(num_sims_per_sigma/5), num_sigmas)
np.save("ws3_norm_no_F_longer.npy", ws__3_norm_no_F)
np.save("optims3_norm_no_F_longer.npy", optims__3_norm_no_F)
np.save("indv_costs3_norm_no_F_longer.npy", individual_costs__3_norm_no_F)
np.save("Ts3_norm_no_F_longer.npy", Ts__3_norm_no_F)

print("Batch 4")
ws__4_norm_no_F, optims__4_norm_no_F, individual_costs__4_norm_no_F, Ts__4_norm_no_F = GENERAL_VAR_INTEGER_OPTIMISE_best(n, N_cells_all_norm[:, 3*int(num_sims_per_sigma/5):4*int(num_sims_per_sigma/5), :], int(num_sims_per_sigma/5), num_sigmas)
np.save("ws4_norm_no_F_longer.npy", ws__4_norm_no_F)
np.save("optims4_norm_no_F_longer.npy", optims__4_norm_no_F)
np.save("indv_costs4_norm_no_F_longer.npy", individual_costs__4_norm_no_F)
np.save("Ts4_norm_no_F_longer.npy", Ts__4_norm_no_F)

print("Batch 5")
ws__5_norm_no_F, optims__5_norm_no_F, individual_costs__5_norm_no_F, Ts__5_norm_no_F = GENERAL_VAR_INTEGER_OPTIMISE_best(n, N_cells_all_norm[:, 4*int(num_sims_per_sigma/5):num_sims_per_sigma, :], int(num_sims_per_sigma/5), num_sigmas)
np.save("ws5_norm_no_F_longer.npy", ws__5_norm_no_F)
np.save("optims5_norm_no_F_longer.npy", optims__5_norm_no_F)
np.save("indv_costs5_norm_no_F_longer.npy", individual_costs__5_norm_no_F)
np.save("Ts5_norm_no_F_longer.npy", Ts__5_norm_no_F)


print("Lognormal distribution...")
print("Batch 1")
ws__1_log_norm, optims__1_log_norm, individual_costs__1_log_norm, Ts__1_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_best(n, N_cells_all_log_norm[:, 0:int(num_sims_per_sigma/5), :], int(num_sims_per_sigma/5), num_sigmas)
np.save("ws1_log_norm_longer.npy", ws__1_log_norm)
np.save("optims1_log_norm_longer.npy", optims__1_log_norm)
np.save("indv_costs1_log_norm_longer.npy", individual_costs__1_log_norm)
np.save("Ts1_log_norm_longer.npy", Ts__1_log_norm)

print("Batch 2")
ws__2_log_norm, optims__2_log_norm, individual_costs__2_log_norm, Ts__2_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_best(n, N_cells_all_log_norm[:, int(num_sims_per_sigma/5):2*int(num_sims_per_sigma/5), :], int(num_sims_per_sigma/5), num_sigmas)
np.save("ws2_log_norm_longer.npy", ws__2_log_norm)
np.save("optims2_log_norm_longer.npy", optims__2_log_norm)
np.save("indv_costs2_log_norm_longer.npy", individual_costs__2_log_norm)
np.save("Ts2_log_norm_longer.npy", Ts__2_log_norm)

print("Batch 3")
ws__3_log_norm, optims__3_log_norm, individual_costs__3_log_norm, Ts__3_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_best(n, N_cells_all_log_norm[:, 2*int(num_sims_per_sigma/5):3*int(num_sims_per_sigma/5), :], int(num_sims_per_sigma/5), num_sigmas)
np.save("ws3_log_norm_longer.npy", ws__3_log_norm)
np.save("optims3_log_norm_longer.npy", optims__3_log_norm)
np.save("indv_costs3_log_norm_longer.npy", individual_costs__3_log_norm)
np.save("Ts3_log_norm_longer.npy", Ts__3_log_norm)

print("Batch 4")
ws__4_log_norm, optims__4_log_norm, individual_costs__4_log_norm, Ts__4_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_best(n, N_cells_all_log_norm[:, 3*int(num_sims_per_sigma/5):4*int(num_sims_per_sigma/5), :], int(num_sims_per_sigma/5), num_sigmas)
np.save("ws4_log_norm_longer.npy", ws__4_log_norm)
np.save("optims4_log_norm_longer.npy", optims__4_log_norm)
np.save("indv_costs4_log_norm_longer.npy", individual_costs__4_log_norm)
np.save("Ts4_log_norm_longer.npy", Ts__4_log_norm)

print("Batch 5")
ws__5_log_norm, optims__5_log_norm, individual_costs__5_log_norm, Ts__5_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_best(n, N_cells_all_log_norm[:, 4*int(num_sims_per_sigma/5):num_sims_per_sigma, :], int(num_sims_per_sigma/5), num_sigmas)
np.save("ws5_log_norm_longer.npy", ws__5_log_norm)
np.save("optims5_log_norm_longer.npy", optims__5_log_norm)
np.save("indv_costs5_log_norm_longer.npy", individual_costs__5_log_norm)
np.save("Ts5_log_norm_longer.npy", Ts__5_log_norm)


In [ ]:
#Opening and combining results
n = 5 #Number of individuals
N_mean = 10000 #Number of cells each individual can occupy
sigma_min = 0 #Lowest SD in foraging abilities considered
sigma_max = 3*N_mean #Highest SD in foraging abilities considered - extreme overdispersion
num_sigmas = 250 #Number of different variances
num_sims_per_sigma = 100 #Number of MC simulations per combination of parameters


#Preallocating space for combined results
individual_costs__norm = [[[None] * num_sims_per_sigma for i in range(num_sigmas)]]
ws__norm = np.zeros([num_sigmas, num_sims_per_sigma, n-1])
Ts__norm = np.zeros([num_sigmas, num_sims_per_sigma])

individual_costs__log_norm = [[[None] * num_sims_per_sigma for i in range(num_sigmas)]]
ws__log_norm = np.zeros([num_sigmas, num_sims_per_sigma, n-1])
Ts__log_norm = np.zeros([num_sigmas, num_sims_per_sigma])

#Loading data - truncated normal distribution
ws__1_norm_no_F = np.load("ws1_norm_no_F_longer.npy")
individual_costs__1_norm_no_F = np.load("indv_costs1_norm_no_F_longer.npy")
Ts__1_norm_no_F = np.load("Ts1_norm_no_F_longer.npy")

ws__2_norm_no_F = np.load("ws2_norm_no_F_longer.npy")
individual_costs__2_norm_no_F = np.load("indv_costs2_norm_no_F_longer.npy")
Ts__2_norm_no_F = np.load("Ts2_norm_no_F_longer.npy")

ws__3_norm_no_F = np.load("ws3_norm_no_F_longer.npy")
individual_costs__3_norm_no_F = np.load("indv_costs3_norm_no_F_longer.npy")
Ts__3_norm_no_F = np.load("Ts3_norm_no_F_longer.npy")

ws__4_norm_no_F = np.load("ws4_norm_no_F_longer.npy")
individual_costs__4_norm_no_F = np.load("indv_costs4_norm_no_F_longer.npy")
Ts__4_norm_no_F = np.load("Ts4_norm_no_F_longer.npy")

ws__5_norm_no_F = np.load("ws5_norm_no_F_longer.npy")
individual_costs__5_norm_no_F = np.load("indv_costs5_norm_no_F_longer.npy")
Ts__5_norm_no_F = np.load("Ts5_norm_no_F_longer.npy")

#Loading data - log-normal distribution
ws__1_log_norm = np.load("ws1_log_norm_longer.npy") #note to self - manually change the name of this file in future to add the "_no_F" suffix, to avoid confusion.

individual_costs__1_log_norm = np.load("indv_costs1_log_norm_longer.npy")
Ts__1_log_norm = np.load("Ts1_log_norm_longer.npy")    

ws__2_log_norm = np.load("ws2_log_norm_longer.npy")
individual_costs__2_log_norm = np.load("indv_costs2_log_norm_longer.npy")
Ts__2_log_norm = np.load("Ts2_log_norm_longer.npy")    

ws__3_log_norm = np.load("ws3_log_norm_longer.npy")
individual_costs__3_log_norm = np.load("indv_costs3_log_norm_longer.npy")
Ts__3_log_norm = np.load("Ts3_log_norm_longer.npy")    

ws__4_log_norm = np.load("ws4_log_norm_longer.npy")
individual_costs__4_log_norm = np.load("indv_costs4_log_norm_longer.npy")
Ts__4_log_norm = np.load("Ts4_log_norm_longer.npy")    

ws__5_log_norm = np.load("ws5_log_norm_longer.npy")
individual_costs__5_log_norm = np.load("indv_costs5_log_norm_longer.npy")
Ts__5_log_norm = np.load("Ts5_log_norm_longer.npy")    

  
# Combining data - truncated normal distribution
#Ws
ws__norm[:, 0:int(num_sims_per_sigma*1/5), :] = ws__1_norm_no_F
ws__norm[:, int(num_sims_per_sigma*1/5):int(num_sims_per_sigma*2/5), :] = ws__2_norm_no_F
ws__norm[:, int(num_sims_per_sigma*2/5):int(num_sims_per_sigma*3/5), :] = ws__3_norm_no_F
ws__norm[:, int(num_sims_per_sigma*3/5):int(num_sims_per_sigma*4/5), :] = ws__4_norm_no_F
ws__norm[:, int(num_sims_per_sigma*4/5):int(num_sims_per_sigma), :] = ws__5_norm_no_F

#Ks
individual_costs__norm[:][:][0:int(num_sims_per_sigma*1/5)] = individual_costs__1_norm_no_F
individual_costs__norm[:][:][int(num_sims_per_sigma*1/5):int(num_sims_per_sigma*2/5)] = individual_costs__2_norm_no_F
individual_costs__norm[:][:][int(num_sims_per_sigma*2/5):int(num_sims_per_sigma*3/5)] = individual_costs__3_norm_no_F
individual_costs__norm[:][:][int(num_sims_per_sigma*3/5):int(num_sims_per_sigma*4/5)] = individual_costs__4_norm_no_F
individual_costs__norm[:][:][int(num_sims_per_sigma*4/5):int(num_sims_per_sigma)] = individual_costs__5_norm_no_F


#Ts
Ts__norm[:, 0:int(num_sims_per_sigma*1/5)] = Ts__1_norm_no_F
Ts__norm[:, int(num_sims_per_sigma*1/5):int(num_sims_per_sigma*2/5)] = Ts__2_norm_no_F
Ts__norm[:, int(num_sims_per_sigma*2/5):int(num_sims_per_sigma*3/5)] = Ts__3_norm_no_F
Ts__norm[:, int(num_sims_per_sigma*3/5):int(num_sims_per_sigma*4/5)] = Ts__4_norm_no_F
Ts__norm[:, int(num_sims_per_sigma*4/5):int(num_sims_per_sigma)] = Ts__5_norm_no_F

# Combining data - log-normal distribution
#Ws
ws__log_norm[:, 0:int(num_sims_per_sigma*1/5), :] = ws__1_log_norm
ws__log_norm[:, int(num_sims_per_sigma*1/5):int(num_sims_per_sigma*2/5), :] = ws__2_log_norm
ws__log_norm[:, int(num_sims_per_sigma*2/5):int(num_sims_per_sigma*3/5), :] = ws__3_log_norm
ws__log_norm[:, int(num_sims_per_sigma*3/5):int(num_sims_per_sigma*4/5), :] = ws__4_log_norm
ws__log_norm[:, int(num_sims_per_sigma*4/5):int(num_sims_per_sigma), :] = ws__5_log_norm

#Ks
individual_costs__log_norm[:][:][0:int(num_sims_per_sigma*1/5)] = individual_costs__1_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*1/5):int(num_sims_per_sigma*2/5)] = individual_costs__2_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*2/5):int(num_sims_per_sigma*3/5)] = individual_costs__3_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*3/5):int(num_sims_per_sigma*4/5)] = individual_costs__4_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*4/5):int(num_sims_per_sigma)] = individual_costs__5_log_norm

#Ts
Ts__log_norm[:, 0:int(num_sims_per_sigma*1/5)] = Ts__1_log_norm
Ts__log_norm[:, int(num_sims_per_sigma*1/5):int(num_sims_per_sigma*2/5)] = Ts__2_log_norm
Ts__log_norm[:, int(num_sims_per_sigma*2/5):int(num_sims_per_sigma*3/5)] = Ts__3_log_norm
Ts__log_norm[:, int(num_sims_per_sigma*3/5):int(num_sims_per_sigma*4/5)] = Ts__4_log_norm
Ts__log_norm[:, int(num_sims_per_sigma*4/5):int(num_sims_per_sigma)] = Ts__5_log_norm

In [ ]:
Ts = Ts__norm
ws = ws__norm
individual_costs = individual_costs__norm

sigmas = np.linspace(sigma_min, sigma_max, num_sigmas)

ws_mean = np.mean(ws, axis=1)
T_vals_mean = np.mean(Ts, axis=1)

y_upper = T_vals_mean + np.std(Ts, axis=1)
y_lower = T_vals_mean - np.std(Ts, axis=1)

y_upper = list(y_upper)
y_lower = list(y_lower)
sigmas_list = list(sigmas)


fig = make_subplots(
        rows=1, cols=2,
        specs=[[{}, {}]], subplot_titles = ["Group information transfer, T", "Average point knowledge, K"])


sigmas = np.linspace(sigma_min, sigma_max, num_sigmas)

fig.add_trace(go.Scatter(
        x=sigmas,
        y=T_vals_mean,
        line=dict(color='rgb(0,100,80)'),
        mode='lines',
        showlegend= False
    ), row=1, col=1)


fig.add_trace(go.Scatter(
        x=sigmas_list+sigmas_list[::-1], # x, then x reversed
        y=y_upper+y_lower[::-1], # upper, then lower reversed
        fill='toself',
        fillcolor='rgba(0,100,80,0.2)',
        line=dict(color='rgba(255,255,255,0)'),
        hoverinfo="skip",
        showlegend=False
    ), row=1, col=1)


Ks = np.zeros((num_sims_per_sigma, num_sigmas))
for j in range(num_sims_per_sigma):
    for i in range(num_sigmas):
        Ks[j, i] = get_w_weight_mean(n, ws[i, j, :])

Ks_mean = np.mean(Ks, axis = 0)

y_upper = Ks_mean + np.std(Ks, axis=0)
y_lower = Ks_mean - np.std(Ks, axis=0)

y_upper = list(y_upper)
y_lower = list(y_lower)

fig.add_trace(go.Scatter(
        x=sigmas,
        y=Ks_mean,
        line=dict(color='rgb(230,97,0)'),
        mode='lines',
        showlegend= False
    ), row=1, col=2)


fig.add_trace(go.Scatter(
        x=sigmas_list+sigmas_list[::-1], # x, then x reversed
        y=y_upper+y_lower[::-1], # upper, then lower reversed
        fill='toself',
        fillcolor='rgba(230,97,0,0.2)',
        line=dict(color='rgba(255,255,255,0)'),
        hoverinfo="skip",
        showlegend=False
    ), row=1, col=2)


for i in [1,2]:
    fig.update_yaxes(linewidth=1, linecolor='black', mirror=True, ticks='inside', 
    showline=True, row = 1, col = i, showgrid = False)
    fig.update_xaxes(linewidth=1, linecolor='black', mirror=True, ticks='inside', 
    showline=True, row = 1, col = i, showgrid = False)
    fig.update_xaxes(title_text=f"SD, sigma",row = 1, col = i)


fig.update_layout(
        autosize=False,
        width=1100,
        height=350)

fig.update_yaxes(title_text=f"Group information transfer, T",row = 1, col = 1)
fig.update_yaxes(title_text=f"Average point knowledge, K",row = 1, col = 1)
fig.write_image("Fig7_top_norm_longer.pdf")

fig.show()


#And then the ws
ws_mean = np.mean(ws, axis=1)

fig = make_subplots(
        rows=1, cols=4)

for i in range(1,5):
    ws_i = ws[:, :, i-1]
    ws_means_i = ws_mean[:, i-1]

    y_upper = ws_means_i + np.std(ws_i, axis=1)
    y_lower = ws_means_i - np.std(ws_i, axis=1)
    y_lower = [max([y_lower[i],0]) for i in range(num_sigmas)] 

    y_upper = list(y_upper)
    y_lower = list(y_lower)
    fig.add_trace(go.Scatter(
        x=sigmas,
        y=ws_means_i,
        line=dict(color='rgb(93,58,155)'),
        mode='lines',
        showlegend= False
    ), row=1, col=i)


    fig.add_trace(go.Scatter(
            x=sigmas_list+sigmas_list[::-1], # x, then x reversed
            y=y_upper+y_lower[::-1], # upper, then lower reversed
            fill='toself',
            fillcolor='rgba(93,58,155,0.2)',
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo="skip",
            showlegend=False
        ), row=1, col=i)

for i in [1,2,3,4]:
    fig.update_yaxes(linewidth=1, linecolor='black', mirror=True, ticks='inside', 
    showline=True, row = 1, col = i, showgrid = False, range = [0, 0.25])
    fig.update_xaxes(linewidth=1, linecolor='black', mirror=True, ticks='inside', 
    showline=True, row = 1, col = i, showgrid = False)
    fig.update_xaxes(title_text=f"SD, sigma",row = 1, col = i) 

fig.update_layout(
        autosize=False,
        width=1100,
        height=350)

fig.write_image("Fig7_bottom_norm_longer.pdf")

fig.show()

Ts = Ts__log_norm
ws = ws__log_norm
individual_costs = individual_costs__log_norm

sigmas = np.linspace(sigma_min, sigma_max, num_sigmas)

ws_mean = np.mean(ws, axis=1)
T_vals_mean = np.mean(Ts, axis=1)

y_upper = T_vals_mean + np.std(Ts, axis=1)
y_lower = T_vals_mean - np.std(Ts, axis=1)

y_upper = list(y_upper)
y_lower = list(y_lower)
sigmas_list = list(sigmas)


fig = make_subplots(
        rows=1, cols=2,
        specs=[[{}, {}]], subplot_titles = ["Group information transfer, T", "Average point knowledge, K"])


sigmas = np.linspace(sigma_min, sigma_max, num_sigmas)

fig.add_trace(go.Scatter(
        x=sigmas,
        y=T_vals_mean,
        line=dict(color='rgb(0,100,80)'),
        mode='lines',
        showlegend= False
    ), row=1, col=1)


fig.add_trace(go.Scatter(
        x=sigmas_list+sigmas_list[::-1], # x, then x reversed
        y=y_upper+y_lower[::-1], # upper, then lower reversed
        fill='toself',
        fillcolor='rgba(0,100,80,0.2)',
        line=dict(color='rgba(255,255,255,0)'),
        hoverinfo="skip",
        showlegend=False
    ), row=1, col=1)


Ks = np.zeros((num_sims_per_sigma, num_sigmas))
for j in range(num_sims_per_sigma):
    for i in range(num_sigmas):
        Ks[j, i] = get_w_weight_mean(n, ws[i, j, :])

Ks_mean = np.mean(Ks, axis = 0)

y_upper = Ks_mean + np.std(Ks, axis=0)
y_lower = Ks_mean - np.std(Ks, axis=0)

y_upper = list(y_upper)
y_lower = list(y_lower)

fig.add_trace(go.Scatter(
        x=sigmas,
        y=Ks_mean,
        line=dict(color='rgb(230,97,0)'),
        mode='lines',
        showlegend= False
    ), row=1, col=2)


fig.add_trace(go.Scatter(
        x=sigmas_list+sigmas_list[::-1], # x, then x reversed
        y=y_upper+y_lower[::-1], # upper, then lower reversed
        fill='toself',
        fillcolor='rgba(230,97,0,0.2)',
        line=dict(color='rgba(255,255,255,0)'),
        hoverinfo="skip",
        showlegend=False
    ), row=1, col=2)


for i in [1,2]:
    fig.update_yaxes(linewidth=1, linecolor='black', mirror=True, ticks='inside', 
    showline=True, row = 1, col = i, showgrid = False)
    fig.update_xaxes(linewidth=1, linecolor='black', mirror=True, ticks='inside', 
    showline=True, row = 1, col = i, showgrid = False)
    fig.update_xaxes(title_text=f"SD, sigma",row = 1, col = i)


fig.update_layout(
        autosize=False,
        width=1100,
        height=350)

fig.update_yaxes(title_text=f"Group information transfer, T",row = 1, col = 1)
fig.update_yaxes(title_text=f"Average point knowledge, K",row = 1, col = 1)
fig.write_image("Fig7_top_lognorm_longer.pdf")

fig.show()


#And then the ws
ws_mean = np.mean(ws, axis=1)

fig = make_subplots(
        rows=1, cols=4)

for i in range(1,5):
    ws_i = ws[:, :, i-1]
    ws_means_i = ws_mean[:, i-1]

    y_upper = ws_means_i + np.std(ws_i, axis=1)
    y_lower = ws_means_i - np.std(ws_i, axis=1)
    y_lower = [max([y_lower[i],0]) for i in range(num_sigmas)] 

    y_upper = list(y_upper)
    y_lower = list(y_lower)
    fig.add_trace(go.Scatter(
        x=sigmas,
        y=ws_means_i,
        line=dict(color='rgb(93,58,155)'),
        mode='lines',
        showlegend= False
    ), row=1, col=i)


    fig.add_trace(go.Scatter(
            x=sigmas_list+sigmas_list[::-1], # x, then x reversed
            y=y_upper+y_lower[::-1], # upper, then lower reversed
            fill='toself',
            fillcolor='rgba(93,58,155,0.2)',
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo="skip",
            showlegend=False
        ), row=1, col=i)

for i in [1,2,3,4]:
    fig.update_yaxes(linewidth=1, linecolor='black', mirror=True, ticks='inside', 
    showline=True, row = 1, col = i, showgrid = False, range = [0, 0.25])
    fig.update_xaxes(linewidth=1, linecolor='black', mirror=True, ticks='inside', 
    showline=True, row = 1, col = i, showgrid = False)
    fig.update_xaxes(title_text=f"SD, sigma",row = 1, col = i) 

fig.update_layout(
        autosize=False,
        width=1100,
        height=350)

fig.write_image("Fig7_bottom_lognorm_longer.pdf")

fig.show()

#### With foraging constraint

##### Optimisation

In [ ]:

def GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_gp(n, N_cells_all, num_sims_per_sigma, num_sigmas, F_min, F_max, num_Fs):
    #Optimise in the case of n individuals with heterogenous abilities, set up to be in order of increasing variance, and a foraging constraint F. Using Gurobi solver.
    with gp.Env(empty=True) as env: #To avoid text showing optimisation procedure. 
        
        env.setParam('OutputFlag', 0)
        env.start()

        #Pre-allocate space and define ranges - note we no longer store the optimal values themselves
        individual_costs = [[[None] * num_sims_per_sigma for i in range(num_sigmas)] for j in range(num_Fs)]
        w_vals = np.zeros([num_Fs, num_sigmas, num_sims_per_sigma, n-1])
        T_vals = np.zeros([num_Fs, num_sigmas, num_sims_per_sigma])
        Fs = np.linspace(F_min, F_max, num_Fs)
        Fs = np.floor(Fs)
        
        #Setup independent to m
        X, N, X_sizes_increasing, X_sizes_decreasing = find_subsets(n)
        
        N_diff = (2**(n-1)) - 1
        entries = np.ones(n*N_diff + N)
        entries[n*N_diff:] = [-(len(X[i]) - 1) for i in range(N)]
        
        row_indices = np.zeros(n * N_diff+N)
        col_indices = [None] * (n * N_diff + N)
        for i in range(n):
            col_indices[i*N_diff:(i+1)*N_diff] = [X.index(set_) for set_ in X if {i}.issubset(set_)]
            row_indices[i*N_diff:(i+1)*N_diff] = i 
    
        col_indices[n*N_diff:] = range(N)
    
        row_indices[n*N_diff:] = n
            
        col_indices = np.array(col_indices)
        
        G = sp.csr_matrix((entries, (row_indices, col_indices)), shape=(n+1, N))
                
        for j in range(num_sims_per_sigma):    
            print(j)
            for i in range(num_sigmas): 
                #First compute for the largest foraging constraint, then work downwards (optimising only when previous solution is infeasible).

                current_space_use = np.inf

                for f_index in list(reversed(range(num_Fs))):
                    F = Fs[f_index] #Current foraging constraint
                    if current_space_use >= F: #optimise

                        #Initialise the model
                        m = gp.Model(env = env, name = "test")

                        
                        #Define the variables (the overlapping points)
                        O = m.addMVar(shape=N, vtype=GRB.INTEGER, name="O") #Constraining the solutions to be integers
                
                        #Adding constraints
                        m.addConstrs((O[i] >= 0 for i in range(N)), name = 'non-neg bounds') #For non-negativity
                        
                        N_cells = N_cells_all[i, j, :]

                        N_cells = [min(F, N_cells[i]) for i in range(n)]
                        
                        #Might want to track the N values themsevles? So we can plot the empirical variances.
                            
                        h = np.array(N_cells)
                        h = np.append(h, [F-sum(N_cells)])
                        
                        m.addConstr(G @ O <= h, name='sharing') #Defining the "don't share too much" constraint
                
                        #Defining the objective function
                        S_sets= [set(S(c, X, X_sizes_decreasing, N)) for c in range(N)]
                        B_sets = [set(B(c, X, X_sizes_increasing, N)) for c in range(N)]
                        sum_Ns, prod_Ns = give_sums_and_prods(X, N, N_cells)
                        l = get_l(N, S_sets, sum_Ns, prod_Ns)
                        M = get_M(X, N, S_sets, B_sets, prod_Ns)
                        
                        m.setObjective(l@O + 0.5 * O.T @ M @ O, GRB.MAXIMIZE)
            
                        #Optimising 
                        m.setParam('TimeLimit', 60*5) #5 mins max getting to a solution
                        m.setParam('MIPGap', 0.001) #Setting a looser bound on optimality
                
                        m.optimize()

                    #Storing data
                    if m.SolCount > 0:
                        optim_Os = O.X
                        T_vals[f_index, i, j] = m.ObjVal
                        individual_costs[f_index][i][j] = get_shared_and_unique(optim_Os, n, N_cells, X)[0]                        
                            
                        for k in range(2, n+1):
                            w_vals[f_index, i, j, k-2] += get_w_i_n(X, optim_Os, N_cells, N, k, n)
                    else: 
                        print(f"No solution found for F={F}, sigma={sigmas[i]}, sim={j}")
                        T_vals[f_index, i, j] = -np.inf #Indicating no solution found with negative infinity, so we can easily go with the other solvers
                        current_space_use = np.inf #Ensuring we try to optimise for the next F value.
                    

                    current_space_use = sum(N_cells) - np.sum([(len(X[i]) - 1)*optim_Os[i] for i in range(N)])

    return w_vals, individual_costs, T_vals

def GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_xp(n, N_cells_all, num_sims_per_sigma, num_sigmas, F_min, F_max, num_Fs):
    #Optimise in the case of n individuals with heterogenous abilities, set up to be in order of increasing variance, and a foraging constraint F. Using Xpress solver.
    xp.controls.outputlog = 0
    xp.setOutputEnabled(False)

    #Pre-allocate space and define ranges - note we no longer store the optimal values themselves.
    individual_costs = [[[None] * num_sims_per_sigma for i in range(num_sigmas)] for j in range(num_Fs)]
    w_vals = np.zeros([num_Fs, num_sigmas, num_sims_per_sigma, n-1])
    T_vals = np.zeros([num_Fs, num_sigmas, num_sims_per_sigma])
    Fs = np.linspace(F_min, F_max, num_Fs)
    Fs = np.floor(Fs)
    
    #Setup independent to m
    X, N, X_sizes_increasing, X_sizes_decreasing = find_subsets(n)
    index_sets = [[X.index(set_) for set_ in X if {i}.issubset(set_)] for i in range(n)]


    #setup involving m, and optimisation
     
    for j in range(num_sims_per_sigma):    
        print(j)
        for i in range(num_sigmas): 
            current_space_use = np.inf
            for f_index in list(reversed(range(num_Fs))):
                F = Fs[f_index] #Current foraging constraint
                if current_space_use >= F: #optimise

                    #Initialise the model
                    m = xp.problem()
                    
                    #Define the variables (the overlapping points)
                    O = np.array([m.addVariable(lb=0, vartype = xp.integer, name = f'O_{i+1}') for i in range(N)]) #Constraining the solutions to be integers
            
                    #Defining the objective
                    
                    N_cells = N_cells_all[i, j, :]

                    N_cells = [min(F, N_cells[i]) for i in range(n)]
                    
                    S_sets= [set(S(c, X, X_sizes_decreasing, N)) for c in range(N)]
                    B_sets = [set(B(c, X, X_sizes_increasing, N)) for c in range(N)]
                    sum_Ns, prod_Ns = give_sums_and_prods(X, N, N_cells)

                    l = get_l(N, S_sets, sum_Ns, prod_Ns)
                    M = get_M(X, N, S_sets, B_sets, prod_Ns)
                        
                    m.setObjective(l@O + 0.5 * O.T @ M @ O, sense = xp.maximize)
        
                    #Adding constraints
                    for a in range(n):
                        m.addConstraint(xp.Sum(O[index_sets[a]]) <= N_cells[a])

                    m.addConstraint(sum(N_cells) - np.sum([(len(X[i]) - 1)*O[i] for i in range(N)]) <= F)   

                    #Optimising 
                    m.controls.timelimit = 180 #3 mins max getting to a solution
                    m.optimize()

                #Storing data
                optim_Os = np.array(m.getSolution())
                T_vals[f_index, i, j] = l@optim_Os + 0.5 * optim_Os.T @ M @ optim_Os
                
                individual_costs[f_index][i][j] = get_shared_and_unique(optim_Os, n, N_cells, X)[0]
                    
                for k in range(2, n+1):
                    w_vals[f_index, i, j, k-2] += get_w_i_n(X, optim_Os, N_cells, N, k, n)

                current_space_use = sum(N_cells) - np.sum([(len(X[i]) - 1)*optim_Os[i] for i in range(N)])


    return w_vals, individual_costs, T_vals

def GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_cplex(n, N_cells_all, num_sims_per_sigma, num_sigmas, F_min, F_max, num_Fs):
    
    #Optimise in the case of n individuals with heterogenous abilities, set up to be in order of increasing variance, and a foraging constraint F. Using CPLEX solver.
    

    #Pre-allocate space and define ranges - note we no longer store the optimal values themselves.
    individual_costs = [[[None] * num_sims_per_sigma for i in range(num_sigmas)] for j in range(num_Fs)]
    w_vals = np.zeros([num_Fs, num_sigmas, num_sims_per_sigma, n-1])
    T_vals = np.zeros([num_Fs, num_sigmas, num_sims_per_sigma])
    Fs = np.linspace(F_min, F_max, num_Fs)
    Fs = np.floor(Fs)
    
    #Setup independent to m
    X, N, X_sizes_increasing, X_sizes_decreasing = find_subsets(n)
    index_sets = [[X.index(set_) for set_ in X if {i}.issubset(set_)] for i in range(n)]
    S_sets= [set(S(c, X, X_sizes_decreasing, N)) for c in range(N)]
    B_sets = [set(B(c, X, X_sizes_increasing, N)) for c in range(N)]


    #setup involving m, and optimisation
    
        
    for j in range(num_sims_per_sigma):    
        print(j)
        for i in range(num_sigmas): 
            current_space_use = np.inf
            for f_index in list(reversed(range(num_Fs))):
                F = Fs[f_index] #Current foraging constraint
                if current_space_use >= F: #optimise
                    #Initialise the model
                    m = cplex.Cplex() 
                
                    m.set_problem_type(m.problem_type.MIQP)
                    m.objective.set_sense(m.objective.sense.maximize)
                    
                    #Define the variables and objective                        
                    N_cells = N_cells_all[i, j, :]
                    N_cells = [min(F, N_cells[i]) for i in range(n)]
                    
                    sum_Ns, prod_Ns = give_sums_and_prods(X, N, N_cells)

                    l = get_l(N, S_sets, sum_Ns, prod_Ns)
                    M = get_M(X, N, S_sets, B_sets, prod_Ns)
                        
                    O = m.variables.add(obj=l, types=[m.variables.type.integer] * N, lb = [0] * N)
                    m.objective.set_quadratic([cplex.SparsePair(ind = range(N), val = [M[i, k] for k in range(N)]) for i in range(N)])

        
                    ##Adding constraints
                    m.linear_constraints.add(
                    lin_expr = [cplex.SparsePair(ind = index_sets[i], val = [1] * len(index_sets[i])) for i in range(n)], 
                    senses = ["L"] * n,
                    rhs = N_cells) #Defining the "don't share too much" constraint

                    m.linear_constraints.add(lin_expr = [cplex.SparsePair(ind = range(N), val = [-(len(X[i]) - 1) for i in range(N)])],
                        senses = ["L"],
                        rhs = [F - sum(N_cells)])

                    m.set_results_stream(None)
                    m.set_log_stream(None)

                    #Optimising 
                    m.parameters.optimalitytarget.set(3)
                    m.parameters.timelimit.set(180) #3 min max 
                    m.set_warning_stream(None)
                    m.set_log_stream(None) 

                    m.solve() 
                    if m.solution.get_status() !=  119 and m.solution.get_status() != 1217: #This is the status code for infeasible solution. Occasionally the solver fails due to its pre-processing methods.
                        optim_Os = np.array(m.solution.get_values())
                        optim_T = m.solution.get_objective_value()
                    else: 
                        optim_Os = np.zeros(N) #Assigning a global minima instead. Other solvers will then overrule this one. 
                        optim_T = -1 #Assigning a value which will NOT be selected instead of the other solvers.

                #Storing data
                T_vals[f_index, i, j] = optim_T
                
                individual_costs[f_index][i][j] = get_shared_and_unique(optim_Os, n, N_cells, X)[0]
                    
                for k in range(2, n+1):
                    w_vals[f_index, i, j, k-2] += get_w_i_n(X, optim_Os, N_cells, N, k, n)

                current_space_use = sum(N_cells) - np.sum([(len(X[i]) - 1)*optim_Os[i] for i in range(N)])

    return w_vals, individual_costs, T_vals

def GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all, num_sims_per_sigma, num_sigmas, F_min, F_max, num_Fs):
    #Optimise with each solver and return the best solution for each tested value of sigma and F
    individual_costs = [[[None] * num_sims_per_sigma for i in range(num_sigmas)] for j in range(num_Fs)]
    ws = np.zeros([num_Fs, num_sigmas, num_sims_per_sigma, n-1])
    Ts = np.zeros([num_Fs, num_sigmas, num_sims_per_sigma])

    T_vals_all = [None] * 3
    ws_all = [None] * 3
    individual_costs_all = [None] * 3

    print("With Gurobi...")
    ws_all[0], individual_costs_all[0], T_vals_all[0] = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_gp(n, N_cells_all, num_sims_per_sigma, num_sigmas, F_min, F_max, num_Fs) 
    print("With Xpress...")
    ws_all[1], individual_costs_all[1], T_vals_all[1] = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_xp(n, N_cells_all, num_sims_per_sigma, num_sigmas, F_min, F_max, num_Fs)
    print("With CPLEX...")
    ws_all[2], individual_costs_all[2], T_vals_all[2] = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_cplex(n, N_cells_all, num_sims_per_sigma, num_sigmas, F_min, F_max, num_Fs)

    print("Comparing results...")
    for f_index in range(num_Fs):
        for i in range(num_sigmas):
            for j in range(num_sims_per_sigma):
                T_vals = [T_vals_all[0][f_index,i,j], T_vals_all[1][f_index,i,j], T_vals_all[2][f_index,i,j]]
                Ts[f_index, i, j] = max(T_vals)
                best_index = T_vals.index(Ts[f_index, i, j])

                individual_costs[f_index][i][j] = individual_costs_all[best_index][f_index][i][j]
                ws[f_index, i, j, :] = ws_all[best_index][f_index, i, j, :]

    
    return ws, individual_costs, Ts

##### Functions for plotting

In [ ]:

def plot_T_with_F_sigma(Ts, sigma_min, sigma_max, num_sigmas, F_min, F_max, num_Fs, N_cell_normal, show_lines = False):
    # Plot contour of T values with F and sigma axes
    Ts_mean = np.mean(Ts, axis = 2)

    Fs = np.linspace(F_min, F_max, num_Fs)
    sigmas = np.linspace(sigma_min, sigma_max, num_sigmas)

    fig = make_subplots(
        rows=1, cols=1)
    
    fig.add_trace(go.Contour(
        z=np.round(Ts_mean,12),
        x=sigmas, 
        y=Fs,
        contours=dict(
            start=0,
            end=np.max(Ts_mean),
            size=np.max(Ts_mean) / 20,
        ), colorscale='viridis'
    ))
    
    fig.update_layout(
    autosize=False,
    width=480,
    height=350,
    margin=dict(
        l=10,
        r=10,
        b=10,
        t=10,
        pad=2
    ))
    
    if show_lines:
        line_grad = N_cell_normal 
        line_intercept = 0
        fig.add_scatter(
            x=[sigma_min, sigma_max], 
            y=[N_cell_normal, N_cell_normal], 
            mode='lines', 
            line_color='red', 
            line_dash = 'dash',
            showlegend=False
        )
        
        fig.add_scatter(
            x=[(F_min - line_intercept) / line_grad, (F_max-c)/line_grad], 
            y=[F_min, F_max], 
            mode='lines', 
            line_color='white', 
            line_dash = 'dash',
            showlegend=False
        )

    fig.update_yaxes(title_text=f"Available resource level, F")
    fig.update_xaxes(title_text=f"Variance, pre-normalisation, sigma")
    
    return fig

def plot_ws_with_F_sigma(n, ws, sigma_min, sigma_max, num_sigmas, F_min, F_max, num_Fs, N_cell_normal, show_lines = False):
    #Plot contour of w_n^{(i)} values with F and sigma axes
    ws_mean = np.mean(ws, axis = 2)

    Fs = np.linspace(F_min, F_max, num_Fs)
    sigmas = np.linspace(sigma_min, sigma_max, num_sigmas)    
    fig = make_subplots(rows=1, cols=n-1)

    fig.update_layout(
        autosize=False,
        width=300*n,
        height=350,
        margin=dict(
            l=10,
            r=10,
            b=10,
            t=10,
            pad=2
        ))
    
    for size in range(2, n+1):
        fig.add_trace(go.Contour(
                z=np.round(ws_mean[:, :, size-2], 10),
                x=sigmas,
                y=Fs, 
                contours=dict(
                    start=0,
                    end=1,
                    size=1 / 20,
                ), line_smoothing=0
            ),row=1, col=size-1)

    
    if show_lines:
        line_grad = N_cell_normal
        line_intercept = 0
        for s in range(1, n):
            fig.add_scatter(
                x=[sigma_min, sigma_max], 
                y=[N_cell_normal, N_cell_normal], 
                mode='lines', 
                line_color='red', 
                line_dash = 'dash',
                showlegend=False, row=1, col=s
            )
            
            fig.add_scatter(
                x=[(F_min - line_intercept ) / line_grad, (F_max-c)/line_grad], 
                y=[F_min, F_max], 
                mode='lines', 
                line_color='white', 
                line_dash = 'dash',
                showlegend=False, row=1, col=s
            )

    fig.update_yaxes(title_text=f"Available resource level, F", row=1, col=1)

    for i in range(1, n):
        fig.update_xaxes(title_text=f"Variance, pre-normalisation, sigma", row=1, col=i)
    

    return fig

def plot_w_mean_general_var_forage(n, ws, sigma_min, sigma_max, num_sigmas, F_min, F_max, num_Fs):
    #Plot contour of K values with F and sigma axes
    ws_mean = np.mean(ws, axis=2)
    Fs = np.linspace(F_min, F_max, num_Fs)
    sigmas = np.linspace(sigma_min, sigma_max, num_sigmas)
    w_means = np.zeros((num_Fs, num_sigmas))

    for i in range(num_sigmas):
        for j in range(num_Fs):
            w_means[j, i] = get_w_weight_mean(n, ws_mean[j, i, :]) #Swapping i and j in second indexing (the one on RHS) - trying to fix bug
            
    
    fig = make_subplots(
        rows=1, cols=1)

    fig.add_trace(go.Contour(
        z=w_means,
        x=sigmas, 
        y=Fs,
        contours=dict(
            start=0,
            end=np.max(w_means),
            size= np.max(w_means) / 20,
        )#, colorscale='viridis'
    ))

    fig.update_layout(
    autosize=False,
    width=480,
    height=350,
    margin=dict(
        l=10,
        r=10,
        b=10,
        t=10,
        pad=2
    ))

    fig.update_yaxes(title_text=f"Available resource level, F")
    fig.update_xaxes(title_text=f"Variance, pre-normalisation, sigma")

    return fig


##### Some results

In [ ]:
n = 5 #Number of individuals
N_mean = 10000 #Number of cells each individual can occupy
sigma_min = 0 #Lowest SD in foraging abilities considered
sigma_max = N_mean #Highest SD in foraging abilities considered
num_sigmas = 50 #Number of different SDs
num_sims_per_sigma = 500 #Number of MC simulations per combination of parameters
F_min = 100 #Lowest number of maximum foraging points considered 
F_max = 5*N_mean #Highest number of maximum foraging points considered 
num_Fs = 50 #Number of different foraging constraints considered

N_cells_all_norm = get_N_cells_norm(n, N_mean, num_sims_per_sigma, sigma_min, sigma_max, num_sigmas, False)
N_cells_all_log_norm = get_N_cells_log_norm(n, N_mean, num_sims_per_sigma, sigma_min, sigma_max, num_sigmas, False)
print(N_cells_all_norm.max(), N_cells_all_log_norm.max())
print(N_cells_all_norm.min(), N_cells_all_log_norm.min())

print("Truncated normal distribution...")
print("Batch 1")
ws__1_norm, individual_costs__1_norm, Ts__1_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_norm[:, 0:int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws1_norm.npy", ws__1_norm)
np.save("indv_costs1_norm.npy", individual_costs__1_norm)
np.save("Ts1_norm.npy", Ts__1_norm)

print("Batch 2")
ws__2_norm, individual_costs__2_norm, Ts__2_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_norm[:, int(num_sims_per_sigma/20):2*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws2_norm.npy", ws__2_norm)
np.save("indv_costs2_norm.npy", individual_costs__2_norm)
np.save("Ts2_norm.npy", Ts__2_norm)

print("Batch 3")
ws__3_norm, individual_costs__3_norm, Ts__3_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_norm[:, 2*int(num_sims_per_sigma/20):3*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws3_norm.npy", ws__3_norm)
np.save("indv_costs3_norm.npy", individual_costs__3_norm)
np.save("Ts3_norm.npy", Ts__3_norm)

print("Batch 4")
ws__4_norm, individual_costs__4_norm, Ts__4_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_norm[:, 3*int(num_sims_per_sigma/20):4*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws4_norm.npy", ws__4_norm)
np.save("indv_costs4_norm.npy", individual_costs__4_norm)
np.save("Ts4_norm.npy", Ts__4_norm)

print("Batch 5")
ws__5_norm, individual_costs__5_norm, Ts__5_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_norm[:, 4*int(num_sims_per_sigma/20):5*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws5_norm.npy", ws__5_norm)
np.save("indv_costs5_norm.npy", individual_costs__5_norm)
np.save("Ts5_norm.npy", Ts__5_norm)

print("Batch 6")
ws__6_norm, individual_costs__6_norm, Ts__6_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_norm[:, 5*int(num_sims_per_sigma/20):6*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws6_norm.npy", ws__6_norm)
np.save("indv_costs6_norm.npy", individual_costs__6_norm)
np.save("Ts6_norm.npy", Ts__6_norm)

print("Batch 7")
ws__7_norm, individual_costs__7_norm, Ts__7_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_norm[:, 6*int(num_sims_per_sigma/20):7*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws7_norm.npy", ws__7_norm)
np.save("indv_costs7_norm.npy", individual_costs__7_norm)
np.save("Ts7_norm.npy", Ts__7_norm)

print("Batch 8")
ws__8_norm, individual_costs__8_norm, Ts__8_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_norm[:, 7*int(num_sims_per_sigma/20):8*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws8_norm.npy", ws__8_norm)
np.save("indv_costs8_norm.npy", individual_costs__8_norm)
np.save("Ts8_norm.npy", Ts__8_norm)

print("Batch 9")
ws__9_norm, individual_costs__9_norm, Ts__9_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_norm[:, 8*int(num_sims_per_sigma/20):9*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws9_norm.npy", ws__9_norm)
np.save("indv_costs9_norm.npy", individual_costs__9_norm)
np.save("Ts9_norm.npy", Ts__9_norm)

print("Batch 10")
ws__10_norm, individual_costs__10_norm, Ts__10_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_norm[:, 9*int(num_sims_per_sigma/20):10*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws10_norm.npy", ws__10_norm)
np.save("indv_costs10_norm.npy", individual_costs__10_norm)
np.save("Ts10_norm.npy", Ts__10_norm)

print("Batch 11")
ws__11_norm, individual_costs__11_norm, Ts__11_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_norm[:, 10*int(num_sims_per_sigma/20):11*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws11_norm.npy", ws__11_norm)
np.save("indv_costs11_norm.npy", individual_costs__11_norm)
np.save("Ts11_norm.npy", Ts__11_norm)

print("Batch 12")
ws__12_norm, individual_costs__12_norm, Ts__12_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_norm[:, 11*int(num_sims_per_sigma/20):12*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws12_norm.npy", ws__12_norm)
np.save("indv_costs12_norm.npy", individual_costs__12_norm)
np.save("Ts12_norm.npy", Ts__12_norm)

print("Batch 13")
ws__13_norm, individual_costs__13_norm, Ts__13_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_norm[:, 12*int(num_sims_per_sigma/20):13*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws13_norm.npy", ws__13_norm)
np.save("indv_costs13_norm.npy", individual_costs__13_norm)
np.save("Ts13_norm.npy", Ts__13_norm)

print("Batch 14")
ws__14_norm, individual_costs__14_norm, Ts__14_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_norm[:, 13*int(num_sims_per_sigma/20):14*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws14_norm.npy", ws__14_norm)
np.save("indv_costs14_norm.npy", individual_costs__14_norm)
np.save("Ts14_norm.npy", Ts__14_norm)

print("Batch 15")
ws__15_norm, individual_costs__15_norm, Ts__15_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_norm[:, 14*int(num_sims_per_sigma/20):15*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws15_norm.npy", ws__15_norm)
np.save("indv_costs15_norm.npy", individual_costs__15_norm)
np.save("Ts15_norm.npy", Ts__15_norm)

print("Batch 16")
ws__16_norm, individual_costs__16_norm, Ts__16_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_norm[:, 15*int(num_sims_per_sigma/20):16*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws16_norm.npy", ws__16_norm)
np.save("indv_costs16_norm.npy", individual_costs__16_norm)
np.save("Ts16_norm.npy", Ts__16_norm)

print("Batch 17")
ws__17_norm, individual_costs__17_norm, Ts__17_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_norm[:, 16*int(num_sims_per_sigma/20):17*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws17_norm.npy", ws__17_norm)
np.save("indv_costs17_norm.npy", individual_costs__17_norm)
np.save("Ts17_norm.npy", Ts__17_norm)

print("Batch 18")
ws__18_norm, individual_costs__18_norm, Ts__18_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_norm[:, 17*int(num_sims_per_sigma/20):18*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws18_norm.npy", ws__18_norm)
np.save("indv_costs18_norm.npy", individual_costs__18_norm)
np.save("Ts18_norm.npy", Ts__18_norm)

print("Batch 19")
ws__19_norm, individual_costs__19_norm, Ts__19_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_norm[:, 18*int(num_sims_per_sigma/20):19*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws19_norm.npy", ws__19_norm)
np.save("indv_costs19_norm.npy", individual_costs__19_norm)
np.save("Ts19_norm.npy", Ts__19_norm)

print("Batch 20")
ws__20_norm, individual_costs__20_norm, Ts__20_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_norm[:, 19*int(num_sims_per_sigma/20):20*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws20_norm.npy", ws__20_norm)
np.save("indv_costs20_norm.npy", individual_costs__20_norm)
np.save("Ts20_norm.npy", Ts__20_norm)




print("Log-normal distribution...")
print("Batch 1")
ws__1_log_norm, individual_costs__1_log_norm, Ts__1_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_log_norm[:, 0*int(num_sims_per_sigma/20):1*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws1_log_norm_withF.npy", ws__1_log_norm)
np.save("indv_costs1_log_norm_withF.npy", individual_costs__1_log_norm)
np.save("Ts1_log_norm_withF.npy", Ts__1_log_norm)                 

print("Batch 2")
ws__2_log_norm, individual_costs__2_log_norm, Ts__2_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_log_norm[:, 1*int(num_sims_per_sigma/20):2*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws2_log_norm_withF.npy", ws__2_log_norm)
np.save("indv_costs2_log_norm_withF.npy", individual_costs__2_log_norm)
np.save("Ts2_log_norm_withF.npy", Ts__2_log_norm)  

print("Batch 3")
ws__3_log_norm, individual_costs__3_log_norm, Ts__3_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_log_norm[:, 2*int(num_sims_per_sigma/20):3*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws3_log_norm_withF.npy", ws__3_log_norm)
np.save("indv_costs3_log_norm_withF.npy", individual_costs__3_log_norm)
np.save("Ts3_log_norm_withF.npy", Ts__3_log_norm)  

print("Batch 4")
ws__4_log_norm, individual_costs__4_log_norm, Ts__4_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_log_norm[:, 3*int(num_sims_per_sigma/20):4*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws4_log_norm_withF.npy", ws__4_log_norm)
np.save("indv_costs4_log_norm_withF.npy", individual_costs__4_log_norm)
np.save("Ts4_log_norm_withF.npy", Ts__4_log_norm)  

print("Batch 5")
ws__5_log_norm, individual_costs__5_log_norm, Ts__5_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_log_norm[:, 4*int(num_sims_per_sigma/20):5*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws5_log_norm_withF.npy", ws__5_log_norm)
np.save("indv_costs5_log_norm_withF.npy", individual_costs__5_log_norm)
np.save("Ts5_log_norm_withF.npy", Ts__5_log_norm)  

print("Batch 6")
ws__6_log_norm, individual_costs__6_log_norm, Ts__6_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_log_norm[:, 5*int(num_sims_per_sigma/20):6*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws6_log_norm_withF.npy", ws__6_log_norm)
np.save("indv_costs6_log_norm_withF.npy", individual_costs__6_log_norm)
np.save("Ts6_log_norm_withF.npy", Ts__6_log_norm)                 

print("Batch 7")
ws__7_log_norm, individual_costs__7_log_norm, Ts__7_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_log_norm[:, 6*int(num_sims_per_sigma/20):7*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws7_log_norm_withF.npy", ws__7_log_norm)
np.save("indv_costs7_log_norm_withF.npy", individual_costs__7_log_norm)
np.save("Ts7_log_norm_withF.npy", Ts__7_log_norm)  

print("Batch 8")
ws__8_log_norm, individual_costs__8_log_norm, Ts__8_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_log_norm[:, 7*int(num_sims_per_sigma/20):8*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws8_log_norm_withF.npy", ws__8_log_norm)
np.save("indv_costs8_log_norm_withF.npy", individual_costs__8_log_norm)
np.save("Ts8_log_norm_withF.npy", Ts__8_log_norm)  

print("Batch 9")
ws__9_log_norm, individual_costs__9_log_norm, Ts__9_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_log_norm[:, 8*int(num_sims_per_sigma/20):9*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws9_log_norm_withF.npy", ws__9_log_norm)
np.save("indv_costs9_log_norm_withF.npy", individual_costs__9_log_norm)
np.save("Ts9_log_norm_withF.npy", Ts__9_log_norm)  

print("Batch 10")
ws__10_log_norm, individual_costs__10_log_norm, Ts__10_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_log_norm[:, 9*int(num_sims_per_sigma/20):10*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws10_log_norm_withF.npy", ws__10_log_norm)
np.save("indv_costs10_log_norm_withF.npy", individual_costs__10_log_norm)
np.save("Ts10_log_norm_withF.npy", Ts__10_log_norm)  

print("Batch 11")
ws__11_log_norm, individual_costs__11_log_norm, Ts__11_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_log_norm[:, 10*int(num_sims_per_sigma/20):11*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws11_log_norm_withF.npy", ws__11_log_norm)
np.save("indv_costs11_log_norm_withF.npy", individual_costs__11_log_norm)
np.save("Ts11_log_norm_withF.npy", Ts__11_log_norm)                 

print("Batch 12")
ws__12_log_norm, individual_costs__12_log_norm, Ts__12_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_log_norm[:, 11*int(num_sims_per_sigma/20):12*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws12_log_norm_withF.npy", ws__12_log_norm)
np.save("indv_costs12_log_norm_withF.npy", individual_costs__12_log_norm)
np.save("Ts12_log_norm_withF.npy", Ts__12_log_norm)  

print("Batch 13")
ws__13_log_norm, individual_costs__13_log_norm, Ts__13_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_log_norm[:, 12*int(num_sims_per_sigma/20):13*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws13_log_norm_withF.npy", ws__13_log_norm)
np.save("indv_costs13_log_norm_withF.npy", individual_costs__13_log_norm)
np.save("Ts13_log_norm_withF.npy", Ts__13_log_norm)  

print("Batch 14")
ws__14_log_norm, individual_costs__14_log_norm, Ts__14_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_log_norm[:, 13*int(num_sims_per_sigma/20):14*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws14_log_norm_withF.npy", ws__14_log_norm)
np.save("indv_costs14_log_norm_withF.npy", individual_costs__14_log_norm)
np.save("Ts14_log_norm_withF.npy", Ts__14_log_norm)  

print("Batch 15")
ws__15_log_norm, individual_costs__15_log_norm, Ts__15_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_log_norm[:, 14*int(num_sims_per_sigma/20):15*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws15_log_norm_withF.npy", ws__15_log_norm)
np.save("indv_costs15_log_norm_withF.npy", individual_costs__15_log_norm)
np.save("Ts15_log_norm_withF.npy", Ts__15_log_norm)  

print("Batch 16")
ws__16_log_norm, individual_costs__16_log_norm, Ts__16_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_log_norm[:, 15*int(num_sims_per_sigma/20):16*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws16_log_norm_withF.npy", ws__16_log_norm)
np.save("indv_costs16_log_norm_withF.npy", individual_costs__16_log_norm)
np.save("Ts16_log_norm_withF.npy", Ts__16_log_norm)                 

print("Batch 17")
ws__17_log_norm, individual_costs__17_log_norm, Ts__17_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_log_norm[:, 16*int(num_sims_per_sigma/20):17*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws17_log_norm_withF.npy", ws__17_log_norm)
np.save("indv_costs17_log_norm_withF.npy", individual_costs__17_log_norm)
np.save("Ts17_log_norm_withF.npy", Ts__17_log_norm)  

print("Batch 18")
ws__18_log_norm, individual_costs__18_log_norm, Ts__18_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_log_norm[:, 17*int(num_sims_per_sigma/20):18*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws18_log_norm_withF.npy", ws__18_log_norm)
np.save("indv_costs18_log_norm_withF.npy", individual_costs__18_log_norm)
np.save("Ts18_log_norm_withF.npy", Ts__18_log_norm)  

print("Batch 19")
ws__19_log_norm, individual_costs__19_log_norm, Ts__19_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_log_norm[:, 18*int(num_sims_per_sigma/20):19*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws19_log_norm_withF.npy", ws__19_log_norm)
np.save("indv_costs19_log_norm_withF.npy", individual_costs__19_log_norm)
np.save("Ts19_log_norm_withF.npy", Ts__19_log_norm)  

print("Batch 20")
ws__20_log_norm, individual_costs__20_log_norm, Ts__20_log_norm = GENERAL_VAR_INTEGER_OPTIMISE_FORAGE_best(n, N_cells_all_log_norm[:, 19*int(num_sims_per_sigma/20):20*int(num_sims_per_sigma/20), :], int(num_sims_per_sigma/20), num_sigmas, F_min, F_max, num_Fs)
np.save("ws20_log_norm_withF.npy", ws__20_log_norm)
np.save("indv_costs20_log_norm_withF.npy", individual_costs__20_log_norm)
np.save("Ts20_log_norm_withF.npy", Ts__20_log_norm)  



In [ ]:
#Pre-allocating space for loading data
num_sigmas = 50 
num_sims_per_sigma = 500
individual_costs__norm = [[[None] * num_sims_per_sigma for i in range(num_sigmas)] for j in range(num_Fs)]
ws__norm = np.zeros([num_Fs, num_sigmas, num_sims_per_sigma, n-1])
Ts__norm = np.zeros([num_Fs, num_sigmas, num_sims_per_sigma])


individual_costs__log_norm = [[[None] * num_sims_per_sigma for i in range(num_sigmas)] for j in range(num_Fs)]
ws__log_norm = np.zeros([num_Fs, num_sigmas, num_sims_per_sigma, n-1])
Ts__log_norm = np.zeros([num_Fs, num_sigmas, num_sims_per_sigma])


#Loading data - truncated normal distribution
ws__1_norm = np.load("ws1_norm.npy")
individual_costs__1_norm = np.load("indv_costs1_norm.npy")
Ts__1_norm = np.load("Ts1_norm.npy")

ws__2_norm = np.load("ws2_norm.npy")
individual_costs__2_norm = np.load("indv_costs2_norm.npy")
Ts__2_norm = np.load("Ts2_norm.npy")

ws__3_norm = np.load("ws3_norm.npy")
individual_costs__3_norm = np.load("indv_costs3_norm.npy")
Ts__3_norm = np.load("Ts3_norm.npy")

ws__4_norm = np.load("ws4_norm.npy")
individual_costs__4_norm = np.load("indv_costs4_norm.npy")
Ts__4_norm = np.load("Ts4_norm.npy")

ws__5_norm = np.load("ws5_norm.npy")
individual_costs__5_norm = np.load("indv_costs5_norm.npy")
Ts__5_norm = np.load("Ts5_norm.npy")

ws__6_norm = np.load("ws6_norm.npy")
individual_costs__6_norm = np.load("indv_costs6_norm.npy")
Ts__6_norm = np.load("Ts6_norm.npy")

ws__7_norm = np.load("ws7_norm.npy")
individual_costs__7_norm = np.load("indv_costs7_norm.npy")
Ts__7_norm = np.load("Ts7_norm.npy")

ws__8_norm = np.load("ws8_norm.npy")
individual_costs__8_norm = np.load("indv_costs8_norm.npy")
Ts__8_norm = np.load("Ts8_norm.npy")

ws__9_norm = np.load("ws9_norm.npy")
individual_costs__9_norm = np.load("indv_costs9_norm.npy")
Ts__9_norm = np.load("Ts9_norm.npy")

ws__10_norm = np.load("ws10_norm.npy")
individual_costs__10_norm = np.load("indv_costs10_norm.npy")
Ts__10_norm = np.load("Ts10_norm.npy")

ws__11_norm = np.load("ws11_norm.npy")
individual_costs__11_norm = np.load("indv_costs11_norm.npy")
Ts__11_norm = np.load("Ts11_norm.npy")

ws__12_norm = np.load("ws12_norm.npy")
individual_costs__12_norm = np.load("indv_costs12_norm.npy")
Ts__12_norm = np.load("Ts12_norm.npy")

ws__13_norm = np.load("ws13_norm.npy")
individual_costs__13_norm = np.load("indv_costs13_norm.npy")
Ts__13_norm = np.load("Ts13_norm.npy")

ws__14_norm = np.load("ws14_norm.npy")
individual_costs__14_norm = np.load("indv_costs14_norm.npy")
Ts__14_norm = np.load("Ts14_norm.npy")

ws__15_norm = np.load("ws15_norm.npy")
individual_costs__15_norm = np.load("indv_costs15_norm.npy")
Ts__15_norm = np.load("Ts15_norm.npy")

ws__16_norm = np.load("ws16_norm.npy")
individual_costs__16_norm = np.load("indv_costs16_norm.npy")
Ts__16_norm = np.load("Ts16_norm.npy")

ws__17_norm = np.load("ws17_norm.npy")
individual_costs__17_norm = np.load("indv_costs17_norm.npy")
Ts__17_norm = np.load("Ts17_norm.npy")

ws__18_norm = np.load("ws18_norm.npy")
individual_costs__18_norm = np.load("indv_costs18_norm.npy")
Ts__18_norm = np.load("Ts18_norm.npy")

ws__19_norm = np.load("ws19_norm.npy")
individual_costs__19_norm = np.load("indv_costs19_norm.npy")
Ts__19_norm = np.load("Ts19_norm.npy")

ws__20_norm = np.load("ws20_norm.npy")
individual_costs__20_norm = np.load("indv_costs20_norm.npy")
Ts__20_norm = np.load("Ts20_norm.npy")


#Loading data - log-normal distribution
ws__1_log_norm = np.load("ws1_log_norm_withF.npy")
individual_costs__1_log_norm = np.load("indv_costs1_log_norm_withF.npy")
Ts__1_log_norm = np.load("Ts1_log_norm_withF.npy")

ws__2_log_norm = np.load("ws2_log_norm_withF.npy")
individual_costs__2_log_norm = np.load("indv_costs2_log_norm_withF.npy")
Ts__2_log_norm = np.load("Ts2_log_norm_withF.npy")

ws__3_log_norm = np.load("ws3_log_norm_withF.npy")
individual_costs__3_log_norm = np.load("indv_costs3_log_norm_withF.npy")
Ts__3_log_norm = np.load("Ts3_log_norm_withF.npy")

ws__4_log_norm = np.load("ws4_log_norm_withF.npy")
individual_costs__4_log_norm = np.load("indv_costs4_log_norm_withF.npy")
Ts__4_log_norm = np.load("Ts4_log_norm_withF.npy")

ws__5_log_norm = np.load("ws5_log_norm_withF.npy")
individual_costs__5_log_norm = np.load("indv_costs5_log_norm_withF.npy")
Ts__5_log_norm = np.load("Ts5_log_norm_withF.npy")

ws__6_log_norm = np.load("ws6_log_norm_withF.npy")
individual_costs__6_log_norm = np.load("indv_costs6_log_norm_withF.npy")
Ts__6_log_norm = np.load("Ts6_log_norm_withF.npy")

ws__7_log_norm = np.load("ws7_log_norm_withF.npy")
individual_costs__7_log_norm = np.load("indv_costs7_log_norm_withF.npy")
Ts__7_log_norm = np.load("Ts7_log_norm_withF.npy")

ws__8_log_norm = np.load("ws8_log_norm_withF.npy")
individual_costs__8_log_norm = np.load("indv_costs8_log_norm_withF.npy")
Ts__8_log_norm = np.load("Ts8_log_norm_withF.npy")

ws__9_log_norm = np.load("ws9_log_norm_withF.npy")
individual_costs__9_log_norm = np.load("indv_costs9_log_norm_withF.npy")
Ts__9_log_norm = np.load("Ts9_log_norm_withF.npy")

ws__10_log_norm = np.load("ws10_log_norm_withF.npy")
individual_costs__10_log_norm = np.load("indv_costs10_log_norm_withF.npy")
Ts__10_log_norm = np.load("Ts10_log_norm_withF.npy")

ws__11_log_norm = np.load("ws11_log_norm_withF.npy")
individual_costs__11_log_norm = np.load("indv_costs11_log_norm_withF.npy")
Ts__11_log_norm = np.load("Ts11_log_norm_withF.npy") 

ws__12_log_norm = np.load("ws12_log_norm_withF.npy")
individual_costs__12_log_norm = np.load("indv_costs12_log_norm_withF.npy")
Ts__12_log_norm = np.load("Ts12_log_norm_withF.npy")

ws__13_log_norm = np.load("ws13_log_norm_withF.npy")
individual_costs__13_log_norm = np.load("indv_costs13_log_norm_withF.npy")
Ts__13_log_norm = np.load("Ts13_log_norm_withF.npy")

ws__14_log_norm = np.load("ws14_log_norm_withF.npy")
individual_costs__14_log_norm = np.load("indv_costs14_log_norm_withF.npy")
Ts__14_log_norm = np.load("Ts14_log_norm_withF.npy")

ws__15_log_norm = np.load("ws15_log_norm_withF.npy")
individual_costs__15_log_norm = np.load("indv_costs15_log_norm_withF.npy")
Ts__15_log_norm = np.load("Ts15_log_norm_withF.npy")

ws__16_log_norm = np.load("ws16_log_norm_withF.npy")
individual_costs__16_log_norm = np.load("indv_costs16_log_norm_withF.npy")
Ts__16_log_norm = np.load("Ts16_log_norm_withF.npy")

ws__17_log_norm = np.load("ws17_log_norm_withF.npy")
individual_costs__17_log_norm = np.load("indv_costs17_log_norm_withF.npy")
Ts__17_log_norm = np.load("Ts17_log_norm_withF.npy")

ws__18_log_norm = np.load("ws18_log_norm_withF.npy")
individual_costs__18_log_norm = np.load("indv_costs18_log_norm_withF.npy")
Ts__18_log_norm = np.load("Ts18_log_norm_withF.npy")

ws__19_log_norm = np.load("ws19_log_norm_withF.npy")
individual_costs__19_log_norm = np.load("indv_costs19_log_norm_withF.npy")
Ts__19_log_norm = np.load("Ts19_log_norm_withF.npy")

ws__20_log_norm = np.load("ws20_log_norm_withF.npy")
individual_costs__20_log_norm = np.load("indv_costs20_log_norm_withF.npy")
Ts__20_log_norm = np.load("Ts20_log_norm_withF.npy")

#Sorry, below is obviously lazy code...
#With truncated normal distribution 
#ws
ws__norm[:, :, 0:int(num_sims_per_sigma*1/20), :] = ws__1_norm
ws__norm[:, :, int(num_sims_per_sigma*1/20):int(num_sims_per_sigma*2/20), :] = ws__2_norm
ws__norm[:, :, int(num_sims_per_sigma*2/20):int(num_sims_per_sigma*3/20), :] = ws__3_norm
ws__norm[:, :, int(num_sims_per_sigma*3/20):int(num_sims_per_sigma*4/20), :] = ws__4_norm
ws__norm[:, :, int(num_sims_per_sigma*4/20):int(num_sims_per_sigma*5/20), :] = ws__5_norm
ws__norm[:, :, int(num_sims_per_sigma*5/20):int(num_sims_per_sigma*6/20), :] = ws__6_norm
ws__norm[:, :, int(num_sims_per_sigma*6/20):int(num_sims_per_sigma*7/20), :] = ws__7_norm
ws__norm[:, :, int(num_sims_per_sigma*7/20):int(num_sims_per_sigma*8/20), :] = ws__8_norm
ws__norm[:, :, int(num_sims_per_sigma*8/20):int(num_sims_per_sigma*9/20), :] = ws__9_norm
ws__norm[:, :, int(num_sims_per_sigma*9/20):int(num_sims_per_sigma*10/20), :] = ws__10_norm
ws__norm[:, :, int(num_sims_per_sigma*10/20):int(num_sims_per_sigma*11/20), :] = ws__11_norm
ws__norm[:, :, int(num_sims_per_sigma*11/20):int(num_sims_per_sigma*12/20), :] = ws__12_norm
ws__norm[:, :, int(num_sims_per_sigma*12/20):int(num_sims_per_sigma*13/20), :] = ws__13_norm
ws__norm[:, :, int(num_sims_per_sigma*13/20):int(num_sims_per_sigma*14/20), :] = ws__14_norm
ws__norm[:, :, int(num_sims_per_sigma*14/20):int(num_sims_per_sigma*15/20), :] = ws__15_norm
ws__norm[:, :, int(num_sims_per_sigma*15/20):int(num_sims_per_sigma*16/20), :] = ws__16_norm
ws__norm[:, :, int(num_sims_per_sigma*16/20):int(num_sims_per_sigma*17/20), :] = ws__17_norm
ws__norm[:, :, int(num_sims_per_sigma*17/20):int(num_sims_per_sigma*18/20), :] = ws__18_norm
ws__norm[:, :, int(num_sims_per_sigma*18/20):int(num_sims_per_sigma*19/20), :] = ws__19_norm
ws__norm[:, :, int(num_sims_per_sigma*19/20):int(num_sims_per_sigma), :] = ws__20_norm


#Ks
individual_costs__norm[:][:][0:int(num_sims_per_sigma*1/20)] = individual_costs__1_norm
individual_costs__norm[:][:][int(num_sims_per_sigma*1/20):int(num_sims_per_sigma*2/20)] = individual_costs__2_norm
individual_costs__norm[:][:][int(num_sims_per_sigma*2/20):int(num_sims_per_sigma*3/20)] = individual_costs__3_norm
individual_costs__norm[:][:][int(num_sims_per_sigma*3/20):int(num_sims_per_sigma*4/20)] = individual_costs__4_norm
individual_costs__norm[:][:][int(num_sims_per_sigma*4/20):int(num_sims_per_sigma*5/20)] = individual_costs__5_norm
individual_costs__norm[:][:][int(num_sims_per_sigma*5/20):int(num_sims_per_sigma*6/20)] = individual_costs__6_norm
individual_costs__norm[:][:][int(num_sims_per_sigma*6/20):int(num_sims_per_sigma*7/20)] = individual_costs__7_norm
individual_costs__norm[:][:][int(num_sims_per_sigma*7/20):int(num_sims_per_sigma*8/20)] = individual_costs__8_norm
individual_costs__norm[:][:][int(num_sims_per_sigma*8/20):int(num_sims_per_sigma*9/20)] = individual_costs__9_norm
individual_costs__norm[:][:][int(num_sims_per_sigma*9/20):int(num_sims_per_sigma*10/20)] = individual_costs__10_norm
individual_costs__norm[:][:][int(num_sims_per_sigma*10/20):int(num_sims_per_sigma*11/20)] = individual_costs__11_norm
individual_costs__norm[:][:][int(num_sims_per_sigma*11/20):int(num_sims_per_sigma*12/20)] = individual_costs__12_norm
individual_costs__norm[:][:][int(num_sims_per_sigma*12/20):int(num_sims_per_sigma*13/20)] = individual_costs__13_norm
individual_costs__norm[:][:][int(num_sims_per_sigma*13/20):int(num_sims_per_sigma*14/20)] = individual_costs__14_norm
individual_costs__norm[:][:][int(num_sims_per_sigma*14/20):int(num_sims_per_sigma*15/20)] = individual_costs__15_norm
individual_costs__norm[:][:][int(num_sims_per_sigma*15/20):int(num_sims_per_sigma*16/20)] = individual_costs__16_norm
individual_costs__norm[:][:][int(num_sims_per_sigma*16/20):int(num_sims_per_sigma*17/20)] = individual_costs__17_norm
individual_costs__norm[:][:][int(num_sims_per_sigma*17/20):int(num_sims_per_sigma*18/20)] = individual_costs__18_norm
individual_costs__norm[:][:][int(num_sims_per_sigma*18/20):int(num_sims_per_sigma*19/20)] = individual_costs__19_norm
individual_costs__norm[:][:][int(num_sims_per_sigma**19/20):int(num_sims_per_sigma)] = individual_costs__20_norm  


#Ts
Ts__norm[:, :, 0:int(num_sims_per_sigma*1/20)] = Ts__1_norm
Ts__norm[:, :, int(num_sims_per_sigma*1/20):int(num_sims_per_sigma*2/20)] = Ts__2_norm
Ts__norm[:, :, int(num_sims_per_sigma*2/20):int(num_sims_per_sigma*3/20)] = Ts__3_norm
Ts__norm[:, :, int(num_sims_per_sigma*3/20):int(num_sims_per_sigma*4/20)] = Ts__4_norm
Ts__norm[:, :, int(num_sims_per_sigma*4/20):int(num_sims_per_sigma*5/20)] = Ts__5_norm
Ts__norm[:, :, int(num_sims_per_sigma*5/20):int(num_sims_per_sigma*6/20)] = Ts__6_norm
Ts__norm[:, :, int(num_sims_per_sigma*6/20):int(num_sims_per_sigma*7/20)] = Ts__7_norm
Ts__norm[:, :, int(num_sims_per_sigma*7/20):int(num_sims_per_sigma*8/20)] = Ts__8_norm
Ts__norm[:, :, int(num_sims_per_sigma*8/20):int(num_sims_per_sigma*9/20)] = Ts__9_norm
Ts__norm[:, :, int(num_sims_per_sigma*9/20):int(num_sims_per_sigma*10/20)] = Ts__10_norm
Ts__norm[:, :, int(num_sims_per_sigma*10/20):int(num_sims_per_sigma*11/20)] = Ts__11_norm
Ts__norm[:, :, int(num_sims_per_sigma*11/20):int(num_sims_per_sigma*12/20)] = Ts__12_norm
Ts__norm[:, :, int(num_sims_per_sigma*12/20):int(num_sims_per_sigma*13/20)] = Ts__13_norm
Ts__norm[:, :, int(num_sims_per_sigma*13/20):int(num_sims_per_sigma*14/20)] = Ts__14_norm
Ts__norm[:, :, int(num_sims_per_sigma*14/20):int(num_sims_per_sigma*15/20)] = Ts__15_norm
Ts__norm[:, :, int(num_sims_per_sigma*15/20):int(num_sims_per_sigma*16/20)] = Ts__16_norm
Ts__norm[:, :, int(num_sims_per_sigma*16/20):int(num_sims_per_sigma*17/20)] = Ts__17_norm
Ts__norm[:, :, int(num_sims_per_sigma*17/20):int(num_sims_per_sigma*18/20)] = Ts__18_norm
Ts__norm[:, :, int(num_sims_per_sigma*18/20):int(num_sims_per_sigma*19/20)] = Ts__19_norm
Ts__norm[:, :, int(num_sims_per_sigma*19/20):int(num_sims_per_sigma)] = Ts__20_norm

#With log-normal distribution
#ws
ws__log_norm[:, :, 0:int(num_sims_per_sigma*1/20), :] = ws__1_log_norm
ws__log_norm[:, :, int(num_sims_per_sigma*1/20):int(num_sims_per_sigma*2/20), :] = ws__2_log_norm
ws__log_norm[:, :, int(num_sims_per_sigma*2/20):int(num_sims_per_sigma*3/20), :] = ws__3_log_norm
ws__log_norm[:, :, int(num_sims_per_sigma*3/20):int(num_sims_per_sigma*4/20), :] = ws__4_log_norm
ws__log_norm[:, :, int(num_sims_per_sigma*4/20):int(num_sims_per_sigma*5/20), :] = ws__5_log_norm
ws__log_norm[:, :, int(num_sims_per_sigma*5/20):int(num_sims_per_sigma*6/20), :] = ws__6_log_norm
ws__log_norm[:, :, int(num_sims_per_sigma*6/20):int(num_sims_per_sigma*7/20), :] = ws__7_log_norm
ws__log_norm[:, :, int(num_sims_per_sigma*7/20):int(num_sims_per_sigma*8/20), :] = ws__8_log_norm
ws__log_norm[:, :, int(num_sims_per_sigma*8/20):int(num_sims_per_sigma*9/20), :] = ws__9_log_norm
ws__log_norm[:, :, int(num_sims_per_sigma*9/20):int(num_sims_per_sigma*10/20), :] = ws__10_log_norm
ws__log_norm[:, :, int(num_sims_per_sigma*10/20):int(num_sims_per_sigma*11/20), :] = ws__11_log_norm
ws__log_norm[:, :, int(num_sims_per_sigma*11/20):int(num_sims_per_sigma*12/20), :] = ws__12_log_norm
ws__log_norm[:, :, int(num_sims_per_sigma*12/20):int(num_sims_per_sigma*13/20), :] = ws__13_log_norm
ws__log_norm[:, :, int(num_sims_per_sigma*13/20):int(num_sims_per_sigma*14/20), :] = ws__14_log_norm
ws__log_norm[:, :, int(num_sims_per_sigma*14/20):int(num_sims_per_sigma*15/20), :] = ws__15_log_norm
ws__log_norm[:, :, int(num_sims_per_sigma*15/20):int(num_sims_per_sigma*16/20), :] = ws__16_log_norm
ws__log_norm[:, :, int(num_sims_per_sigma*16/20):int(num_sims_per_sigma*17/20), :] = ws__17_log_norm
ws__log_norm[:, :, int(num_sims_per_sigma*17/20):int(num_sims_per_sigma*18/20), :] = ws__18_log_norm
ws__log_norm[:, :, int(num_sims_per_sigma*18/20):int(num_sims_per_sigma*19/20), :] = ws__19_log_norm
ws__log_norm[:, :, int(num_sims_per_sigma*19/20):int(num_sims_per_sigma), :] = ws__20_log_norm

#Ks
individual_costs__log_norm[:][:][0:int(num_sims_per_sigma*1/20)] = individual_costs__1_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*1/20):int(num_sims_per_sigma*2/20)] = individual_costs__2_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*2/20):int(num_sims_per_sigma*3/20)] = individual_costs__3_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*3/20):int(num_sims_per_sigma*4/20)] = individual_costs__4_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*4/20):int(num_sims_per_sigma*5/20)] = individual_costs__5_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*5/20):int(num_sims_per_sigma*6/20)] = individual_costs__6_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*6/20):int(num_sims_per_sigma*7/20)] = individual_costs__7_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*7/20):int(num_sims_per_sigma*8/20)] = individual_costs__8_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*8/20):int(num_sims_per_sigma*9/20)] = individual_costs__9_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*9/20):int(num_sims_per_sigma*10/20)] = individual_costs__10_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*10/20):int(num_sims_per_sigma*11/20)] = individual_costs__11_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*11/20):int(num_sims_per_sigma*12/20)] = individual_costs__12_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*12/20):int(num_sims_per_sigma*13/20)] = individual_costs__13_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*13/20):int(num_sims_per_sigma*14/20)] = individual_costs__14_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*14/20):int(num_sims_per_sigma*15/20)] = individual_costs__15_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*15/20):int(num_sims_per_sigma*16/20)] = individual_costs__16_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*16/20):int(num_sims_per_sigma*17/20)] = individual_costs__17_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*17/20):int(num_sims_per_sigma*18/20)] = individual_costs__18_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*18/20):int(num_sims_per_sigma*19/20)] = individual_costs__19_log_norm
individual_costs__log_norm[:][:][int(num_sims_per_sigma*19/20):int(num_sims_per_sigma*20/20)] = individual_costs__20_log_norm


#Ts
Ts__log_norm[:, :, 0:int(num_sims_per_sigma*1/20)] = Ts__1_log_norm
Ts__log_norm[:, :, int(num_sims_per_sigma*1/20):int(num_sims_per_sigma*2/20)] = Ts__2_log_norm
Ts__log_norm[:, :, int(num_sims_per_sigma*2/20):int(num_sims_per_sigma*3/20)] = Ts__3_log_norm
Ts__log_norm[:, :, int(num_sims_per_sigma*3/20):int(num_sims_per_sigma*4/20)] = Ts__4_log_norm
Ts__log_norm[:, :, int(num_sims_per_sigma*4/20):int(num_sims_per_sigma*5/20)] = Ts__5_log_norm
Ts__log_norm[:, :, int(num_sims_per_sigma*5/20):int(num_sims_per_sigma*6/20)] = Ts__6_log_norm
Ts__log_norm[:, :, int(num_sims_per_sigma*6/20):int(num_sims_per_sigma*7/20)] = Ts__7_log_norm
Ts__log_norm[:, :, int(num_sims_per_sigma*7/20):int(num_sims_per_sigma*8/20)] = Ts__8_log_norm
Ts__log_norm[:, :, int(num_sims_per_sigma*8/20):int(num_sims_per_sigma*9/20)] = Ts__9_log_norm
Ts__log_norm[:, :, int(num_sims_per_sigma*9/20):int(num_sims_per_sigma*10/20)] = Ts__10_log_norm
Ts__log_norm[:, :, int(num_sims_per_sigma*10/20):int(num_sims_per_sigma*11/20)] = Ts__11_log_norm
Ts__log_norm[:, :, int(num_sims_per_sigma*11/20):int(num_sims_per_sigma*12/20)] = Ts__12_log_norm
Ts__log_norm[:, :, int(num_sims_per_sigma*12/20):int(num_sims_per_sigma*13/20)] = Ts__13_log_norm
Ts__log_norm[:, :, int(num_sims_per_sigma*13/20):int(num_sims_per_sigma*14/20)] = Ts__14_log_norm
Ts__log_norm[:, :, int(num_sims_per_sigma*14/20):int(num_sims_per_sigma*15/20)] = Ts__15_log_norm
Ts__log_norm[:, :, int(num_sims_per_sigma*15/20):int(num_sims_per_sigma*16/20)] = Ts__16_log_norm
Ts__log_norm[:, :, int(num_sims_per_sigma*16/20):int(num_sims_per_sigma*17/20)] = Ts__17_log_norm
Ts__log_norm[:, :, int(num_sims_per_sigma*17/20):int(num_sims_per_sigma*18/20)] = Ts__18_log_norm
Ts__log_norm[:, :, int(num_sims_per_sigma*18/20):int(num_sims_per_sigma*19/20)] = Ts__19_log_norm
Ts__log_norm[:, :, int(num_sims_per_sigma*19/20):int(num_sims_per_sigma*20/20)] = Ts__20_log_norm


In [ ]:
want_lines = False

sigma_min = 0
sigma_max = N_mean
num_sigmas = 50
F_min = 100
F_max = 50000
num_Fs = 50

print("With truncated normal distribution...")
ws__ = ws__norm
individual_costs__ = individual_costs__norm
Ts__ = Ts__norm

fig = plot_T_with_F_sigma(Ts__, sigma_min, sigma_max, num_sigmas, F_min, F_max, num_Fs, N_mean, want_lines)
fig.show()


fig = plot_ws_with_F_sigma(n, ws__, sigma_min, sigma_max, num_sigmas, F_min, F_max, num_Fs, N_mean, want_lines)
fig.update_layout(
        autosize=False,
        width=1500,
        height=350,
        margin=dict(
            
            l=10,
            r=10,
            b=10,
            t=25,
            pad=2
        ))
fig.write_image("Fig8_ws_normal_500MC.pdf")
fig.show()


fig = plot_w_mean_general_var_forage(n, ws__, sigma_min, sigma_max, num_sigmas, F_min, F_max, num_Fs)
fig.show()

print("With log-normal distribution...")
ws__ = ws__log_norm
individual_costs__ = individual_costs__log_norm
Ts__ = Ts__log_norm

fig = plot_T_with_F_sigma(Ts__, sigma_min, sigma_max, num_sigmas, F_min, F_max, num_Fs, N_mean, want_lines)
fig.show()

fig = plot_ws_with_F_sigma(n, ws__, sigma_min, sigma_max, num_sigmas, F_min, F_max, num_Fs, N_mean, want_lines)
fig.update_layout(
        autosize=False,
        width=1500,
        height=350,
    
        margin=dict(
            
            l=10,
            r=10,
            b=10,
            t=25,
            pad=2
        ))
fig.write_image("Fig8_ws_lognorm_500MC.pdf")

fig.show()

fig = plot_w_mean_general_var_forage(n, ws__, sigma_min, sigma_max, num_sigmas, F_min, F_max, num_Fs)
fig.show()


In [ ]:
print("With truncated normal distribution...")
ws__ = ws__norm
individual_costs__ = individual_costs__norm
Ts__ = Ts__norm


Ts_mean = np.mean(Ts__, axis = 2)

Fs = np.linspace(F_min, F_max, num_Fs)
sigmas = np.linspace(sigma_min, sigma_max, num_sigmas)

fig = make_subplots(
        rows=1, cols=2,
        specs=[[{}, {}]], subplot_titles = ["Group information transfer, T", "Average point knowledge, K"])


fig.add_trace(go.Contour(
        z=np.round(Ts_mean,12),
        x=sigmas, 
        y=Fs,
        contours=dict(
            start=0,
            end=7.5,
            size=6 / 20,
        ), colorscale='viridis'
    ), col = 1, row = 1)


ws_mean = np.mean(ws__, axis=2)

    
w_means = np.zeros((num_Fs, num_sigmas))

for i in range(num_sigmas):
    for j in range(num_Fs):
        w_means[i, j] = get_w_weight_mean(n, ws_mean[i, j, :])

fig.add_trace(go.Contour(
        z=w_means,
        x=sigmas, 
        y=Fs,
        contours=dict(
            start=1,
            end=n,
            size= (n-1) / 20,
        ), colorscale='oranges'
    ), col = 2, row = 1)


fig.update_yaxes(title_text=f"Available resource level, F")
fig.update_xaxes(title_text=f"Variance, pre-normalisation, sigma")

fig.update_layout(
        autosize=False,
        width=1500,
        height=350,
        margin=dict(
            l=10,
            r=10,
            b=10,
            t=25,
            pad=2
        ))

fig.write_image("Fig8normal_500MC.pdf")
fig.show()



print("With log-normal distribution...")
ws__ = ws__log_norm
individual_costs__ = individual_costs__log_norm
Ts__ = Ts__log_norm

Ts_mean = np.mean(Ts__, axis = 2)

Fs = np.linspace(F_min, F_max, num_Fs)
sigmas = np.linspace(sigma_min, sigma_max, num_sigmas)

fig = make_subplots(
        rows=1, cols=2,
        specs=[[{}, {}]], subplot_titles = ["Group information transfer, T", "Average point knowledge, K"])


fig.add_trace(go.Contour(
        z=np.round(Ts_mean,12),
        x=sigmas, 
        y=Fs,
        contours=dict(
            start=0,
            end=7.5,
            size=6 / 20,
        ), colorscale='viridis'
    ), col = 1, row = 1)


ws_mean = np.mean(ws__, axis=2)

    
w_means = np.zeros((num_Fs, num_sigmas))

for i in range(num_sigmas):
    for j in range(num_Fs):
        w_means[i, j] = get_w_weight_mean(n, ws_mean[i, j, :])

fig.add_trace(go.Contour(
        z=w_means,
        x=sigmas, 
        y=Fs,
        contours=dict(
            start=1,
            end=n,
            size= (n-1) / 20,
        ), colorscale='oranges'
    ), col = 2, row = 1)


fig.update_yaxes(title_text=f"Available resource level, F")
fig.update_xaxes(title_text=f"Variance, pre-normalisation, sigma")

fig.update_layout(
        autosize=False,
        width=1500,
        height=350,
        margin=dict(
            l=10,
            r=10,
            b=10,
            t=25,
            pad=2
        ))

fig.write_image("Fig8_top_lognormal_500MC.pdf")
fig.show()
